# Отбор признаков `nsel` — E-Cup Ozon 2026

Финальная версия отбора признаков по схеме трёх фолдов (требования от 28.08.2026).
Ноутбук содержит **ровно тот же код**, что и пакет в папке `kod/`, файл в файл.
Отличие только в том, что перед каждым модулем стоит пояснение.

**Почему код записывается на диск через `%%writefile`, а не выполняется прямо в ячейках.**
Обучения идут в пуле процессов, поднятом способом `spawn`. Такой процесс-исполнитель
стартует с чистого интерпретатора и импортирует функцию задания по имени модуля.
Функцию, объявленную в ячейке ноутбука, он не найдёт. Поэтому ноутбук сначала
раскладывает пакет в папку `nsel_iz_noutbuka/`, потом импортирует его, и дальше всё
работает так же, как при запуске из командной строки.

## Содержание

1. Общая схема отбора
2. Код пакета: конфигурация → данные → модели → исполнитель → кэш и пул → стадии → оркестратор
3. Результаты боевых прогонов (12 сочетаний модели и таргета)
4. Проверка на синтетике: 100 пользователей, 27 признаков, из них 20 шумовых
5. Как запустить на настоящих данных
6. Сверка: код из ноутбука совпадает с `kod/`

## 1. Общая схема

```
срезы parquet по датам отсечения
      │  data.prepare_snapshots — один раз раскладывает в .npy (признак × строка)
      ▼
стадия 1  null importance        какие признаки важнее, чем при перемешанном таргете
стадия 3  ranked forward         добавляем признаки по рангу, правило приёма двух путей
стадия 5  парное спасение        пробуем пары почти прошедших кандидатов
стадия 4  проверка на сидах      весь набор на сидах 42/52/62 (+ опц. разбор сомнительных)
стадия 6  обратная чистка        ровно одна, в самом конце
стадия 7  Optuna (опционально)   на той же кросс-валидации
      ▼
selected_features_<модель>_<таргет>.json
```

Стадии 2 (Group LOFO) нет: её вырезали по требованиям от 28.08, а `checks.py` проверяет,
что она не запускалась. Номера стадий сохранились от прежней схемы, поэтому пятая
идёт раньше четвёртой.

**Кросс-валидация.** Три временных фолда, каждый следующий учится на всех предыдущих датах:

| фолд | обучение (даты отсечения) | проверка | вес |
|---|---|---|---:|
| nov_2025 | 16.09, 16.10 | 15.11 | 0.25 |
| dec_2025 | 16.09, 16.10, 15.11 | 15.12 | 0.35 |
| jan_2026 | 16.09, 16.10, 15.11, 15.12 | 14.01 | 0.40 |

Метрика набора — взвешенная сумма RMSLE по фолдам.

**Правило приёма кандидата в стадии 3:**

```
обычный путь:      выигрыш >= 0.00005 и улучшено фолдов >= 2
устойчивый путь:   выигрыш >= 0.00001 и улучшено фолдов == 3
защита (на обоих): просадка фолда <= 0.005, сегмента нулей <= 0.02, покупавших <= 0.02
```

**Инженерная часть, которая не меняет решений, а только ускоряет счёт:**

* *Фолды не держатся в памяти.* Срезы один раз переписываются в `.npy`, каждый процесс
  открывает их через `mmap` (общие страницы на всю машину), а матрица фолда собирается
  по столбцам только на время одного обучения (`data.py`).
* *Батчи.* Работа режется на задания «обучить набор на одном фолде» (`jobs.py`) и
  считается пулом процессов (`pool.py`). Для стадии 1 пул широкий (мало процессов,
  много потоков), для остальных узкий.
* *Кэш* в sqlite (`cache.py`) на трёх уровнях: каждое обучение по ключу из всех
  параметров, состояние стадии после каждого решения, итог стадии. Обрыв на любом
  месте не теряет посчитанного, повторный запуск продолжает с того же места.
* *Счёт на опережение* (`pool.iterate_in_order`). Пока решается судьба кандидата N,
  кандидаты N+1, N+2… уже считаются. Ширина опережения удваивается на каждом отказе
  до потолка 16. Принятие кандидата обрывает очередь ровно там, где оборвался бы
  последовательный проход, поэтому итог тот же самый.

In [1]:
from pathlib import Path

PKG_ROOT = Path("nsel_iz_noutbuka")
(PKG_ROOT / "nsel" / "stages").mkdir(parents=True, exist_ok=True)
(PKG_ROOT / "tools").mkdir(parents=True, exist_ok=True)
print("пакет раскладывается в", PKG_ROOT.resolve())

пакет раскладывается в /Volumes/ECupOzon/tmp/nsel_otbor_priznakov_20260830/nsel_iz_noutbuka


## 2. Код пакета

### `nsel/__init__.py` — описание пакета

Только строка документации и список модулей.

In [2]:
%%writefile nsel_iz_noutbuka/nsel/__init__.py
"""Отбор признаков по схеме коллеги от 28 августа 2026.

Три временных фолда, стадии 1/3/4/5 без Group LOFO, двухуровневое правило
приёма, обратная чистка ровно один раз в конце. Логика повторяет его
требования дословно; параллельность и кэш добавлены так, чтобы решения
совпадали с последовательным проходом.
"""
__all__ = ["config", "data", "encoding", "model", "cache", "pool", "orchestrator"]

Writing nsel_iz_noutbuka/nsel/__init__.py


### `nsel/config.py` — контракт фолдов, пороги стадий, параметры прогона

Всё, что влияет на решения отбора, собрано здесь в неизменяемых dataclass'ах.

* `FOLD_CONTRACT` и `assert_fold_contract` — три фолда и их веса. Проверяется, что
  между датой обучения и датой проверки не меньше 30 дней, то есть таргет обучения не
  заглядывает в окно проверки.
* `TARGET_MODES` — в каком пространстве учится модель: `log_level` — это `log1p(y)`,
  остальные считают отклонение от якоря. У адаптивных таргетов якорь лежит в отдельном
  файле (`ADAPTIVE_ANCHORS`), у `segment_logspace` он уже в логарифме (`LOG_ANCHOR_MODES`).
* `ProbeConfig` — модель-щуп. Она сравнивает наборы между собой, финальную модель не строит.
  `learning_rate=0.15` вместо боевых 0.03: ранговая корреляция с медленным щупом 0.9998,
  а считается в 2.5 раза быстрее.
* `Stage1Config … BackwardConfig` — пороги стадий (см. схему выше).
* `ParallelConfig` — сколько процессов и потоков. Число процессов выбирается по ядрам
  и свободной памяти, если не задано явно.
* `RunConfig.fingerprint()` — отпечаток всех решающих настроек. По нему кэш понимает,
  что сохранённый итог стадии относится к этим же правилам. Ширина опережения в отпечаток
  не входит, поэтому прогон продолжается и на машине с другим числом ядер.
* `smoke_config` — быстрый прогон: меньше объём счёта, правила те же.

In [3]:
%%writefile nsel_iz_noutbuka/nsel/config.py
"""Конфигурация отбора: контракт фолдов, пороги стадий, параметры Оптуны."""
from __future__ import annotations

import dataclasses
import datetime
import hashlib
import json
import os
from pathlib import Path

HORIZON_DAYS = 30

# Новое требование: три фолда вместо пяти, февраль и октябрь сняты.
FOLD_CONTRACT = (
    ("nov_2025", ("2025-09-16", "2025-10-16"), "2025-11-15", 0.25),
    ("dec_2025", ("2025-09-16", "2025-10-16", "2025-11-15"), "2025-12-15", 0.35),
    ("jan_2026", ("2025-09-16", "2025-10-16", "2025-11-15", "2025-12-15"),
     "2026-01-14", 0.40),
)
EXPECTED_FOLD_WEIGHTS = {"nov_2025": 0.25, "dec_2025": 0.35, "jan_2026": 0.40}

MODEL_MODES = ("catboost", "lightgbm", "xgboost")
TARGET_MODES = ("log_level", "log_diff", "segment_adaptive",
                "volatility_adaptive", "personal_hard", "segment_logspace",
                "log_ratio_ewma")
# У log_ratio_ewma якорь обычный, он уже лежит в срезах отдельным столбцом,
# считать его не нужно.
SNAPSHOT_ANCHORS = {"log_ratio_ewma": "gmv_decay30_hl30"}
# Адаптивные якоря не лежат в срезах, они считаются из сырой истории функцией
# коллеги и кладутся рядом файлом, см. tools/build_adaptive_anchor.py
ADAPTIVE_ANCHORS = {"segment_adaptive": "anchor_segment_adaptive",
                    "volatility_adaptive": "anchor_volatility_adaptive",
                    "personal_hard": "anchor_personal_hard",
                    "segment_logspace": "anchor_segment_logspace_log"}
# У segment_logspace якорь уже в логарифме, поэтому таргет считается как
# log1p(y) - anchor, без второго логарифма. Обычная формула дала бы молча
# другой таргет и неверную метрику.
LOG_ANCHOR_MODES = ("segment_logspace",)

TARGET_ANCHOR_COLUMN = "gmv_30d"
TARGET_COLUMN = "target_30d"
SNAPSHOT_INDEX_COLUMN = "__snapshot_index_user_id__"


class ContractError(AssertionError):
    """Нарушение контракта валидации."""


@dataclasses.dataclass(frozen=True)
class FoldSpec:
    """Один временной фолд единой кросс-валидации."""

    name: str
    train_cutoffs: tuple
    valid_cutoff: str
    weight: float

    def key_parts(self) -> tuple:
        """Части ключа кэша, описывающие фолд."""
        return (self.name, tuple(self.train_cutoffs), self.valid_cutoff,
                round(self.weight, 12))

    def as_dict(self) -> dict:
        """Представление фолда, пригодное для передачи в процесс-воркер."""
        return {"name": self.name, "train_cutoffs": list(self.train_cutoffs),
                "valid_cutoff": self.valid_cutoff, "weight": self.weight}


def build_folds(contract=FOLD_CONTRACT) -> list:
    """Строит список фолдов по контракту."""
    return [FoldSpec(name, tuple(cutoffs), valid, float(weight))
            for name, cutoffs, valid, weight in contract]


def assert_fold_contract(folds: list) -> None:
    """Проверяет состав, веса и отсутствие заглядывания в будущее."""
    if len(folds) != 3:
        raise ContractError(f"фолдов {len(folds)}, по новой схеме должно быть три")
    names = [fold.name for fold in folds]
    if names != list(EXPECTED_FOLD_WEIGHTS):
        raise ContractError(f"состав фолдов {names}")
    total = sum(fold.weight for fold in folds)
    if abs(total - 1.0) > 1e-12:
        raise ContractError(f"веса дают {total}, а должны единицу")
    for fold in folds:
        if abs(fold.weight - EXPECTED_FOLD_WEIGHTS[fold.name]) > 1e-12:
            raise ContractError(f"{fold.name}: вес {fold.weight}")
        valid = datetime.date.fromisoformat(fold.valid_cutoff)
        if valid.year == 2025 and valid.month in (2, 10):
            raise ContractError(f"{fold.name}: февраль и октябрь сняты с проверки")
        for cutoff in fold.train_cutoffs:
            gap = (valid - datetime.date.fromisoformat(cutoff)).days
            if gap < HORIZON_DAYS:
                raise ContractError(
                    f"{fold.name}: между {cutoff} и {fold.valid_cutoff} {gap} дней")


@dataclasses.dataclass(frozen=True)
class ProbeConfig:
    """Модель-щуп: сравнивает наборы между собой, а не строит финальную модель."""

    # 0.15 против боевых 0.03 по замеру research/REPORT.md: ранговая
    # корреляция 0.9998, десять лучших наборов совпали, счёт быстрее в 2.5 раза.
    learning_rate: float = 0.15
    depth: int = 5
    l2_leaf_reg: float = 3.0
    # Потолок деревьев и ранняя остановка — его числа: auto_extend выключен,
    # ceiling 8000, остановка через 30 итераций без улучшения вместо 50.
    iterations: int = 8000
    early_stopping_rounds: int = 30
    subsample: float = 0.8
    colsample: float = 0.8
    min_data_in_leaf: int = 20


@dataclasses.dataclass(frozen=True)
class Stage1Config:
    """Временная null importance."""

    enabled: bool = True
    actual_runs: int = 3
    null_runs: int = 15
    quantile: float = 0.90
    min_keep: int = 250
    max_keep: int = 450
    fallback_top_fraction: float = 0.15
    # У null-модели перемешан и проверочный таргет тоже, поэтому ранняя
    # остановка ничего не знает о настоящей цели и утечки не создаёт. Так же
    # устроено у коллеги, и именно поэтому его null-прогоны дёшевы: без сигнала
    # обучение обрывается через несколько десятков деревьев.
    null_iterations: int = 8000
    null_early_stopping: int = 30
    actual_iterations: int = 8000
    actual_early_stopping: int = 30


@dataclasses.dataclass(frozen=True)
class Stage3Config:
    """Ranked forward selection с двухуровневым правилом приёма."""

    # Его комментарий над аргументом: проверить надо все признаки, пережившие
    # null importance, дополнительным top-K фильтром их не резать. Ноль здесь
    # и означает «без ограничения».
    top_k_candidates: int = 0
    rerank_after_n_added: int = 10
    min_improvement: float = 0.00005
    min_improved_folds: int = 2
    stable_min_improvement: float = 0.00001
    stable_min_improved_folds: int = 3
    # Защиты ослаблены его правкой: не душить признаки, полезные в одном
    # режиме, guard остаётся только против аномального провала.
    max_allowed_fold_degradation: float = 0.005
    max_zero_segment_degradation: float = 0.02
    max_positive_segment_degradation: float = 0.02
    max_features_to_add: int = 10 ** 9
    patience_no_improvement: int = 10 ** 9
    # Потолок опережения. Фактическая ширина растёт от единицы вдвое на каждом
    # отказе: пока приёмы часты, считать далеко вперёд — выбрасывать работу.
    speculation_width: int = 16


@dataclasses.dataclass(frozen=True)
class Stage4Config:
    """Multiseed: сиды проверяют набор целиком, а не каждый признак."""

    enabled: bool = True
    seeds: tuple = (42, 52, 62)
    confirm_doubtful: bool = False
    doubtful_improvement_factor: float = 2.0
    # Признак, прошедший путём устойчивого малого выигрыша, на точечную
    # проверку не отправляется: он и так входит в проверку набора целиком.
    confirm_stable_tiny_gains: bool = False
    # Достаточно, чтобы польза подтвердилась на двух сидах из трёх и медиана
    # была положительной. Никакого дополнительного порога: задача проверки —
    # знак и устойчивость, а не повторное применение порога приёма.
    confirmation_min_positive_seeds: int = 2
    confirmation_min_median_improvement: float = 0.0
    # Метрики сегментов на дополнительных сидах только логируются и права
    # отклонить признак не имеют.
    segment_mode: str = "diagnostic"


@dataclasses.dataclass(frozen=True)
class Stage5Config:
    """Парное спасение почти прошедших кандидатов."""

    enabled: bool = True
    top_rejected: int = 6
    max_pairs: int = 15
    max_additions: int = 2
    # Порог пары полтора порога одиночного признака, а не четыре: спасение ищет
    # умеренные эффекты взаимодействия, требовать от них вчетверо больше нечем.
    min_improvement: float = 0.000075
    min_improved_folds: int = 2
    max_allowed_fold_degradation: float = 0.005
    near_miss_fraction: float = 0.2
    prefer_cross_family: bool = True


@dataclasses.dataclass(frozen=True)
class BackwardConfig:
    """Обратная чистка: ровно один раз, в конце."""

    enabled: bool = True
    neutral_tolerance: float = 0.000005
    speculation_width: int = 16


@dataclasses.dataclass(frozen=True)
class EncodingConfig:
    """Сглаживание ordered target encoding для XGBoost и LightGBM."""

    mode: str = "empirical_bayes"
    fixed_smoothing: float = 20.0
    min_count: int = 1
    first_block_value: str = "nan"
    unknown_value: str = "global"
    n_blocks: int = 20
    minimum_variance: float = 1e-9
    maximum_smoothing: float = 10000.0


@dataclasses.dataclass(frozen=True)
class OptunaConfig:
    """Параметры адаптивного поиска из второго промпта."""

    enabled: bool = False
    adaptive_search: bool = True
    exploration_trials: int = 210
    focused_trials: int = 500
    tpe_startup_trials_explore: int = 20
    tpe_startup_trials_focus: int = 15
    tpe_multivariate: bool = True
    tpe_group: bool = True
    # Раздел 3 промпта пересматривает эти числа: приоритет — не потерять
    # возможный оптимум, поэтому границы берутся по минимуму и максимуму
    # лучших двадцати точек, а не по децилям.
    adaptive_top_k: int = 20
    adaptive_range_expand: float = 0.25
    adaptive_min_range_fraction: float = 0.30
    adaptive_boundary_guard_fraction: float = 0.10
    adaptive_quantile_low: float = 0.00
    adaptive_quantile_high: float = 1.00
    adaptive_keep_categorical_neighbors: int = 1
    adaptive_narrow_binary_categoricals: bool = False
    adaptive_seed_focus_with_best: bool = True
    early_stopping_rounds: int = 50
    gpu: bool = False
    devices: str = "0"
    fallback_to_cpu: bool = False
    strict_ozon_fold_contract: bool = True
    random_state: int = 42


@dataclasses.dataclass(frozen=True)
class ParallelConfig:
    """Занятость машины: сколько процессов и сколько потоков каждому."""

    n_workers: int = 0
    threads_per_worker: int = 2
    memory_per_worker_gb: float = 1.5
    # Стадия 1 учит на всём наборе признаков: замер на сервере дал пик 11 ГБ на
    # процесс против гигабайта на остальных стадиях. Один пул на весь прогон
    # означал бы, что самая прожорливая стадия задаёт размер всем остальным.
    # Замер на пустой машине: одно обучение стадии 1 стоит 0.84 секунды на
    # дерево, а под десятью параллельными процессами — шестнадцать. Упирается
    # не в ядра, а в полосу памяти: каждый процесс прокачивает матрицу на
    # четыре гигабайта. Поэтому здесь мало процессов и много потоков каждому.
    wide_threads_per_worker: int = 10
    wide_memory_per_worker_gb: float = 14.0
    reserve_gb: float = 8.0

    def threads_for(self, wide: bool) -> int:
        """Сколько потоков даётся одному обучению."""
        if wide:
            return self.wide_threads_per_worker
        return self.threads_per_worker

    def memory_for(self, wide: bool) -> float:
        """Сколько памяти закладывается на один процесс."""
        if wide:
            return self.wide_memory_per_worker_gb
        return self.memory_per_worker_gb

    def resolve_workers(self, cpu_count: int, available_gb: float,
                        wide: bool = False) -> int:
        """Число процессов пула по ядрам и свободной памяти."""
        if self.n_workers > 0:
            return self.n_workers
        by_cores = max(1, (cpu_count - 2) // max(1, self.threads_for(wide)))
        by_memory = int((available_gb - self.reserve_gb) / self.memory_for(wide))
        return max(1, min(by_cores, max(1, by_memory)))


@dataclasses.dataclass(frozen=True)
class RunConfig:
    """Полное описание одного прогона отбора."""

    run_dir: Path
    snapshot_dir: Path
    model_mode: str = "catboost"
    target_mode: str = "log_level"
    random_state: int = 42
    users_limit: int = 0
    feature_limit: int = 0
    # Разложенные срезы зависят только от каталога данных и объёма выборки,
    # поэтому шесть сочетаний делят один каталог: иначе он копируется шесть раз
    # по четыре с половиной гигабайта и раскладывается шесть раз подряд.
    shared_prepared_dir: str = ""
    # Категории из биннинга счётчиков он просил отключить целиком.
    drop_count_bin_categories: bool = True
    # Каталог с посчитанным якорём segment_adaptive; для остальных таргетов пуст.
    anchor_dir: str = ""
    probe: ProbeConfig = dataclasses.field(default_factory=ProbeConfig)
    stage1: Stage1Config = dataclasses.field(default_factory=Stage1Config)
    stage3: Stage3Config = dataclasses.field(default_factory=Stage3Config)
    stage4: Stage4Config = dataclasses.field(default_factory=Stage4Config)
    stage5: Stage5Config = dataclasses.field(default_factory=Stage5Config)
    backward: BackwardConfig = dataclasses.field(default_factory=BackwardConfig)
    encoding: EncodingConfig = dataclasses.field(default_factory=EncodingConfig)
    optuna: OptunaConfig = dataclasses.field(default_factory=OptunaConfig)
    parallel: ParallelConfig = dataclasses.field(default_factory=ParallelConfig)

    def __post_init__(self) -> None:
        """Отклоняет неизвестную модель или таргет до начала счёта."""
        if self.model_mode not in MODEL_MODES:
            raise ValueError(f"model_mode={self.model_mode}")
        if self.target_mode not in TARGET_MODES:
            raise ValueError(f"target_mode={self.target_mode}")
        if self.target_mode in ADAPTIVE_ANCHORS and not self.anchor_dir:
            raise ValueError(f"{self.target_mode} требует anchor_dir с якорем")

    @property
    def combination(self) -> str:
        """Имя сочетания модели и таргета."""
        return f"{self.model_mode}_{self.target_mode}"

    @property
    def cache_db(self) -> Path:
        """Путь до базы кэша итераций и стадий."""
        return Path(self.run_dir) / "cache.sqlite3"

    @property
    def log_file(self) -> Path:
        """Путь до подробного текстового лога."""
        return Path(self.run_dir) / "run.log"

    @property
    def events_file(self) -> Path:
        """Путь до машиночитаемого журнала событий."""
        return Path(self.run_dir) / "events.jsonl"

    @property
    def prepared_dir(self) -> Path:
        """Каталог разложенных срезов, общий для всех сочетаний."""
        if self.shared_prepared_dir:
            return Path(self.shared_prepared_dir)
        return Path(self.run_dir) / "prepared"

    def decision_payload(self) -> dict:
        """Всё, что влияет на решения отбора."""
        payload = {
            "folds": [fold.key_parts() for fold in build_folds()],
            "model_mode": self.model_mode,
            "target_mode": self.target_mode,
            "random_state": self.random_state,
            "users_limit": self.users_limit,
            "feature_limit": self.feature_limit,
            "probe": dataclasses.asdict(self.probe),
            "stage1": dataclasses.asdict(self.stage1),
            "stage3": dataclasses.asdict(self.stage3),
            "stage4": dataclasses.asdict(self.stage4),
            "stage5": dataclasses.asdict(self.stage5),
            "backward": dataclasses.asdict(self.backward),
            "encoding": dataclasses.asdict(self.encoding),
        }
        # Ширина спекуляции меняет только порядок счёта, поэтому в отпечаток не
        # входит: прогон должен продолжаться на машине с другим числом ядер.
        payload["stage3"].pop("speculation_width")
        payload["backward"].pop("speculation_width")
        return payload

    def fingerprint(self) -> str:
        """Отпечаток решающих настроек прогона."""
        blob = json.dumps(self.decision_payload(), sort_keys=True,
                          default=str).encode("utf-8")
        return hashlib.sha256(blob).hexdigest()[:16]


def smoke_config(run_dir, snapshot_dir, **overrides) -> RunConfig:
    """Быстрый прогон на настоящих данных: меньше объём, те же правила."""
    base = dict(
        run_dir=Path(run_dir),
        snapshot_dir=Path(snapshot_dir),
        users_limit=8000,
        feature_limit=90,
        probe=ProbeConfig(iterations=300, early_stopping_rounds=25),
        stage1=Stage1Config(actual_runs=2, null_runs=6, min_keep=20, max_keep=60,
                            null_iterations=60, actual_iterations=200,
                            actual_early_stopping=25),
        stage3=Stage3Config(top_k_candidates=24, max_features_to_add=8,
                            speculation_width=6),
        parallel=ParallelConfig(threads_per_worker=2),
    )
    base.update(overrides)
    return RunConfig(**base)


def config_from_env(run_dir, snapshot_dir) -> RunConfig:
    """Боевой конфиг с точечными правками из переменных окружения."""
    kwargs = {"run_dir": Path(run_dir), "snapshot_dir": Path(snapshot_dir),
              "model_mode": os.environ.get("NSEL_MODEL", "catboost"),
              "target_mode": os.environ.get("NSEL_TARGET", "log_level")}
    if os.environ.get("NSEL_USERS"):
        kwargs["users_limit"] = int(os.environ["NSEL_USERS"])
    if os.environ.get("NSEL_FEATURES"):
        kwargs["feature_limit"] = int(os.environ["NSEL_FEATURES"])
    if os.environ.get("NSEL_WORKERS") or os.environ.get("NSEL_THREADS"):
        kwargs["parallel"] = ParallelConfig(
            n_workers=int(os.environ.get("NSEL_WORKERS", "0")),
            threads_per_worker=int(os.environ.get("NSEL_THREADS", "2")))
    if os.environ.get("NSEL_PROBE_LR"):
        kwargs["probe"] = ProbeConfig(learning_rate=float(os.environ["NSEL_PROBE_LR"]))
    if os.environ.get("NSEL_PAIR_RESCUE") == "1":
        kwargs["stage5"] = Stage5Config(enabled=True)
    if os.environ.get("NSEL_CONFIRM_DOUBTFUL") == "1":
        kwargs["stage4"] = Stage4Config(confirm_doubtful=True)
    if os.environ.get("NSEL_OPTUNA") == "1":
        kwargs["optuna"] = OptunaConfig(
            enabled=True, gpu=os.environ.get("NSEL_OPTUNA_GPU", "0") == "1")
    return RunConfig(**kwargs)

Writing nsel_iz_noutbuka/nsel/config.py


### `nsel/logging_setup.py` — журналы

Подробный текстовый лог `run.log` и машиночитаемый `events.jsonl`: по одной строке
на каждое решение о кандидате и каждый конец стадии.

In [4]:
%%writefile nsel_iz_noutbuka/nsel/logging_setup.py
"""Логирование прогона: подробный текст на диск и события в JSONL."""
from __future__ import annotations

import json
import logging
import sys
import time
from pathlib import Path

_EVENTS_PATH = {"path": None}


def setup_logging(log_file, events_file, level=logging.INFO) -> logging.Logger:
    """Настраивает логгер прогона и путь журнала событий."""
    Path(log_file).parent.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger("nsel")
    logger.setLevel(level)
    logger.handlers.clear()
    formatter = logging.Formatter(
        "%(asctime)s %(levelname)-7s %(message)s", datefmt="%H:%M:%S")
    file_handler = logging.FileHandler(log_file, encoding="utf-8")
    file_handler.setFormatter(formatter)
    stream_handler = logging.StreamHandler(sys.stdout)
    stream_handler.setFormatter(formatter)
    logger.addHandler(file_handler)
    logger.addHandler(stream_handler)
    logger.propagate = False
    _EVENTS_PATH["path"] = Path(events_file)
    Path(events_file).parent.mkdir(parents=True, exist_ok=True)
    return logger


def emit_event(kind: str, **fields) -> None:
    """Дописывает одно машиночитаемое событие в журнал."""
    path = _EVENTS_PATH["path"]
    if path is None:
        return
    record = {"time": time.time(), "kind": kind}
    record.update(fields)
    with open(path, "a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")

Writing nsel_iz_noutbuka/nsel/logging_setup.py


### `nsel/data.py` — раскладка срезов и сборка фолдов (здесь решена проблема памяти)

**Подготовка (`prepare_snapshots`) делается один раз на прогон.**
1. Читает схемы `snapshot_YYYYMMDD.parquet` по всем нужным датам и берёт признаки,
   которые есть во всех срезах. Категории из биннинга счётчиков (`*_cat`, кроме
   календарных `cutoff_*`) отбрасывает.
2. Для каждой даты пишет `num_<дата>.npy` (числовые признаки, матрица «признак × строка»,
   float32), `cat_<дата>.npy` (коды категорий из словаря, общего для всех срезов),
   `y_<дата>.npy` (таргет) и `anchor_<дата>.npy` (якорь).
3. Пишет `manifest.json` с подписью. Если подпись совпадает, повторная подготовка
   пропускается.

**Сборка фолда (`materialize`) идёт в каждом процессе-исполнителе.**
* `_memory_map` открывает `.npy` с `mmap_mode="r"`. Страницы файла общие для всех
  процессов, и в память процесса копируется только то, что реально читается.
* `_build_matrix` и `_build_frame` собирают матрицу по одному столбцу сразу в итоговый
  массив, без промежуточной матрицы «строка × признак». На миллионе строк и 900
  признаках промежуточная копия стоила бы ещё 3.5–4 ГБ на процесс.
* CatBoost получает `DataFrame` с целыми кодами категорий, XGBoost и LightGBM — матрицу.

Ещё здесь: `stable_user_mask` (детерминированная подвыборка пользователей по хэшу),
`temporal_shuffle` (перемешивание таргета внутри временного блока, нужно стадии 1),
`feature_family` (семья признака без хвоста-окна, нужна парному спасению).

In [5]:
%%writefile nsel_iz_noutbuka/nsel/data.py
"""Раскладка срезов в numpy и материализация фолдов."""
from __future__ import annotations

import hashlib
import json
import logging
from pathlib import Path

import numpy as np

from .config import (
    ADAPTIVE_ANCHORS,
    SNAPSHOT_ANCHORS,
    SNAPSHOT_INDEX_COLUMN,
    TARGET_ANCHOR_COLUMN,
    TARGET_COLUMN,
)

SERVICE_COLUMNS = {SNAPSHOT_INDEX_COLUMN, TARGET_COLUMN}
NUMERIC_PREFIXES = ("int", "float", "double", "uint")
CALENDAR_CATEGORY_PREFIX = "cutoff_"
CATEGORY_SUFFIX = "_cat"


def is_count_bin_category(name: str) -> bool:
    """Категория, построенная биннингом счётчика активности."""
    if not name.endswith(CATEGORY_SUFFIX):
        return False
    return not name.startswith(CALENDAR_CATEGORY_PREFIX)


def snapshot_path(snapshot_dir, cutoff: str) -> Path:
    """Путь до parquet-среза на дату отсечения."""
    return Path(snapshot_dir) / f"snapshot_{cutoff.replace('-', '')}.parquet"


def required_cutoffs(folds: list) -> list:
    """Все даты отсечения, нужные набору фолдов."""
    dates = set()
    for fold in folds:
        dates.update(fold.train_cutoffs)
        dates.add(fold.valid_cutoff)
    return sorted(dates)


def _hash_users(user_ids) -> np.ndarray:
    """Устойчивый хэш идентификаторов пользователей."""
    return np.array(
        [int(hashlib.blake2b(str(value).encode(), digest_size=8).hexdigest(), 16)
         for value in user_ids],
        dtype=np.uint64,
    )


def stable_user_mask(user_ids, limit: int) -> np.ndarray:
    """Детерминированная подвыборка пользователей по хэшу."""
    if limit <= 0 or limit >= len(user_ids):
        return np.ones(len(user_ids), dtype=bool)
    digest = _hash_users(user_ids)
    order = np.argsort(digest, kind="stable")
    mask = np.zeros(len(user_ids), dtype=bool)
    mask[order[:limit]] = True
    return mask


MINIMUM_CATEGORICAL_IN_SUBSET = 3


def _sample_names(pool: list, count: int, seed: int) -> set:
    """Детерминированная выборка заданного размера из списка имён."""
    if count <= 0 or not pool:
        return set()
    generator = np.random.RandomState(seed)
    chosen = generator.choice(len(pool), size=min(count, len(pool)), replace=False)
    return {pool[index] for index in chosen}


def choose_feature_subset(names: list, categorical: list, limit: int,
                          seed: int = 42) -> list:
    """Подмножество признаков для укороченного прогона с долей категориальных."""
    if limit <= 0 or limit >= len(names):
        return list(names)
    keep = set()
    if TARGET_ANCHOR_COLUMN in names:
        keep.add(TARGET_ANCHOR_COLUMN)
    # Категориальные держим в выборке принудительно: иначе быстрый прогон не
    # проверял бы кодирование, а именно оно ломается тише всего.
    categorical_pool = [name for name in categorical if name not in keep]
    keep |= _sample_names(categorical_pool,
                          min(MINIMUM_CATEGORICAL_IN_SUBSET, max(0, limit - 1)), seed)
    numeric_pool = [name for name in names
                    if name not in keep and name not in set(categorical)]
    keep |= _sample_names(numeric_pool, limit - len(keep), seed + 1)
    return [name for name in names if name in keep]


def _column_kinds(schema, feature_names: list) -> tuple:
    """Делит признаки на числовые и категориальные по схеме parquet."""
    types = dict(zip(schema.names, schema.types))
    categorical = [name for name in feature_names
                   if not str(types[name]).startswith(NUMERIC_PREFIXES)]
    numeric = [name for name in feature_names if name not in set(categorical)]
    return numeric, categorical


def _shared_feature_names(schemas: dict, cutoffs: list, feature_limit: int,
                          drop_count_bins: bool = True) -> list:
    """Признаки, присутствующие во всех срезах, в порядке первого среза."""
    common = set(schemas[cutoffs[0]].names)
    for cutoff in cutoffs[1:]:
        common &= set(schemas[cutoff].names)
    names = [name for name in schemas[cutoffs[0]].names
             if name in common and name not in SERVICE_COLUMNS]
    # Категории из биннинга счётчиков он просил отключить: не показали пользы.
    # Календарные категории остаются, их он оставил включёнными.
    if drop_count_bins:
        names = [name for name in names if not is_count_bin_category(name)]
    _numeric, categorical = _column_kinds(schemas[cutoffs[0]], names)
    return choose_feature_subset(names, categorical, feature_limit)


def _encode_categoricals(frame, categorical: list, tables: dict) -> np.ndarray:
    """Кодирует категориальные столбцы общим для всех срезов словарём."""
    import polars as pl

    codes = np.empty((len(categorical), frame.height), dtype=np.int32)
    for row, name in enumerate(categorical):
        values = frame[name].cast(pl.Utf8).fill_null("__nan__").to_list()
        table = tables[name]
        for position, value in enumerate(values):
            code = table.get(value)
            if code is None:
                code = len(table)
                table[value] = code
            codes[row, position] = code
    return codes


def _write_numeric(frame, numeric: list, destination: Path) -> None:
    """Пишет числовые признаки матрицей «признак x строка»."""
    matrix = np.empty((len(numeric), frame.height), dtype=np.float32)
    for row, name in enumerate(numeric):
        column = frame[name].to_numpy().astype(np.float32, copy=False)
        matrix[row, :] = np.where(np.isfinite(column), column, np.nan)
    np.save(destination, matrix)


def _write_adaptive_anchor(frame, cutoff: str, prepared_dir: Path,
                           anchor_dir) -> None:
    """Кладёт адаптивный якорь в порядке строк среза.

    Порядок задаётся самим срезом, поэтому якорь не приклеивается по позиции, а
    сопоставляется по пользователю: молчаливый сдвиг на одну строку испортил бы
    весь таргет и не проявился бы ни в одной проверке.
    """
    import polars as pl

    source = Path(anchor_dir) / (
        f"anchor_segment_adaptive_{cutoff.replace('-', '')}.parquet")
    if not source.exists():
        raise FileNotFoundError(f"нет файла якоря {source}")
    anchors = pl.read_parquet(source).rename(
        {"user_id": SNAPSHOT_INDEX_COLUMN})
    joined = frame.select(SNAPSHOT_INDEX_COLUMN).join(
        anchors, on=SNAPSHOT_INDEX_COLUMN, how="left")
    for column in sorted(set(ADAPTIVE_ANCHORS.values())):
        if column not in joined.columns:
            raise KeyError(f"в файле якоря {source.name} нет столбца {column}")
        values = joined[column].to_numpy().astype(np.float64)
        missing = int(np.isnan(values).sum())
        if missing:
            raise ValueError(f"якорь {column} за {cutoff}: не нашлось "
                             f"{missing} пользователей из {len(values)}")
        np.save(prepared_dir / f"{column}_{cutoff}.npy", values)


def _prepare_one_cutoff(source: Path, cutoff: str, prepared_dir: Path,
                        columns: list, numeric: list, categorical: list,
                        tables: dict, users_limit: int, anchor_dir=None) -> int:
    """Раскладывает один срез в файлы numpy и возвращает число строк."""
    import polars as pl

    frame = pl.read_parquet(source, columns=columns).sort(SNAPSHOT_INDEX_COLUMN)
    mask = stable_user_mask(frame[SNAPSHOT_INDEX_COLUMN].to_numpy(), users_limit)
    if not mask.all():
        frame = frame.filter(pl.Series(mask))
    _write_numeric(frame, numeric, prepared_dir / f"num_{cutoff}.npy")
    np.save(prepared_dir / f"cat_{cutoff}.npy",
            _encode_categoricals(frame, categorical, tables))
    target = np.nan_to_num(frame[TARGET_COLUMN].to_numpy().astype(np.float64), nan=0.0)
    np.save(prepared_dir / f"y_{cutoff}.npy", target)
    anchor = np.nan_to_num(
        frame[TARGET_ANCHOR_COLUMN].to_numpy().astype(np.float64), nan=0.0)
    np.save(prepared_dir / f"anchor_{cutoff}.npy", anchor)
    if anchor_dir:
        _write_adaptive_anchor(frame, cutoff, prepared_dir, anchor_dir)
    for column in sorted(set(SNAPSHOT_ANCHORS.values())):
        if column not in frame.columns:
            continue
        values = np.nan_to_num(
            frame[column].to_numpy().astype(np.float64), nan=0.0)
        np.save(prepared_dir / f"{column}_{cutoff}.npy", values)
    return frame.height


def prepare_snapshots(snapshot_dir, prepared_dir, cutoffs, users_limit=0,
                      feature_limit=0, logger=None, force=False,
                      drop_count_bins=True, anchor_dir=None) -> dict:
    """Раскладывает срезы в numpy один раз и возвращает манифест."""
    import pyarrow.parquet as pq

    log = logger or logging.getLogger("nsel.data")
    prepared_dir = Path(prepared_dir)
    prepared_dir.mkdir(parents=True, exist_ok=True)
    cutoffs = list(cutoffs)
    signature = hashlib.sha256(json.dumps(
        {"cutoffs": cutoffs, "users": users_limit, "features": feature_limit,
         "dir": str(snapshot_dir), "drop_count_bins": drop_count_bins,
         "anchor_dir": str(anchor_dir or "")},
        sort_keys=True).encode()).hexdigest()[:16]

    manifest_path = prepared_dir / "manifest.json"
    if manifest_path.exists() and not force:
        manifest = json.loads(manifest_path.read_text())
        if manifest.get("signature") == signature:
            log.info("[подготовка] срезы уже разложены, подпись %s", signature)
            return manifest

    schemas = {}
    for cutoff in cutoffs:
        path = snapshot_path(snapshot_dir, cutoff)
        if not path.exists():
            raise FileNotFoundError(f"нет среза {path}")
        schemas[cutoff] = pq.ParquetFile(path).schema_arrow

    feature_names = _shared_feature_names(schemas, cutoffs, feature_limit,
                                          drop_count_bins)
    numeric, categorical = _column_kinds(schemas[cutoffs[0]], feature_names)
    columns = [SNAPSHOT_INDEX_COLUMN, TARGET_COLUMN] + feature_names
    if TARGET_ANCHOR_COLUMN not in columns:
        columns.append(TARGET_ANCHOR_COLUMN)
    for extra in sorted(set(SNAPSHOT_ANCHORS.values())):
        if extra not in columns:
            columns.append(extra)
    log.info("[подготовка] признаков %d: числовых %d, категориальных %d",
             len(feature_names), len(numeric), len(categorical))

    tables = {name: {} for name in categorical}
    rows = {}
    for cutoff in cutoffs:
        log.info("[подготовка] раскладываю срез %s", cutoff)
        rows[cutoff] = _prepare_one_cutoff(
            snapshot_path(snapshot_dir, cutoff), cutoff, prepared_dir, columns,
            numeric, categorical, tables, users_limit, anchor_dir)

    manifest = {"signature": signature, "cutoffs": cutoffs, "numeric": numeric,
                "categorical": categorical, "features": feature_names, "rows": rows,
                "cardinality": {name: len(table) for name, table in tables.items()},
                "prepared_dir": str(prepared_dir)}
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2))
    log.info("[подготовка] готово: срезов %d, строк на срез %s",
             len(cutoffs), sorted(set(rows.values())))
    return manifest


_MEMORY_MAPS: dict = {}
_MANIFESTS: dict = {}


def load_manifest(prepared_dir: str) -> dict:
    """Читает манифест подготовки, кэшируя его в процессе."""
    if prepared_dir not in _MANIFESTS:
        _MANIFESTS[prepared_dir] = json.loads(
            (Path(prepared_dir) / "manifest.json").read_text())
    return _MANIFESTS[prepared_dir]


def _memory_map(prepared_dir: str, name: str) -> np.ndarray:
    """Открывает файл numpy отображением в память, разделяемым между процессами."""
    key = f"{prepared_dir}/{name}"
    if key not in _MEMORY_MAPS:
        _MEMORY_MAPS[key] = np.load(Path(prepared_dir) / name, mmap_mode="r")
    return _MEMORY_MAPS[key]


def _column_values(prepared_dir: str, cutoffs: list, prefix: str, row: int,
                   dtype) -> np.ndarray:
    """Значения одного признака, склеенные по срезам обучения."""
    parts = [np.asarray(_memory_map(prepared_dir, f"{prefix}_{cutoff}.npy")[row, :],
                        dtype=dtype) for cutoff in cutoffs]
    if len(parts) == 1:
        return parts[0]
    return np.concatenate(parts)


def _build_frame(prepared_dir: str, cutoffs: list, numeric_names: list,
                 numeric_rows: list, categorical_names: list,
                 categorical_rows: list):
    """Собирает таблицу по одному столбцу за раз, без промежуточной матрицы.

    Промежуточная матрица «строка x признак» удваивала бы память: на миллионе
    строк и девятистах признаках это лишние четыре гигабайта на процесс, и
    пятнадцать процессов клали машину по памяти.
    """
    import pandas as pd

    columns = {}
    for name, row in zip(numeric_names, numeric_rows):
        columns[name] = _column_values(prepared_dir, cutoffs, "num", row, np.float32)
    for name, row in zip(categorical_names, categorical_rows):
        columns[name] = _column_values(prepared_dir, cutoffs, "cat", row, np.int32)
    return pd.DataFrame(columns, columns=numeric_names + categorical_names,
                        copy=False)


def _build_matrix(prepared_dir: str, cutoffs: list, numeric_rows: list,
                  categorical_rows: list) -> np.ndarray:
    """Собирает матрицу «строка x признак» сразу, без промежуточных блоков.

    Раньше блоки по срезам собирались отдельно и потом склеивались, и матрица
    жила в памяти дважды: на миллионе строк и девятистах признаках это лишние
    три с половиной гигабайта на процесс, из-за которых в память влезало
    вдвое меньше процессов.
    """
    manifest = load_manifest(prepared_dir)
    heights = [manifest["rows"][cutoff] for cutoff in cutoffs]
    width = len(numeric_rows) + len(categorical_rows)
    matrix = np.empty((sum(heights), width), dtype=np.float32)
    offset = 0
    for cutoff, height in zip(cutoffs, heights):
        column = 0
        for row in numeric_rows:
            source = _memory_map(prepared_dir, f"num_{cutoff}.npy")[row, :]
            matrix[offset:offset + height, column] = source
            column += 1
        for row in categorical_rows:
            source = _memory_map(prepared_dir, f"cat_{cutoff}.npy")[row, :]
            matrix[offset:offset + height, column] = source
            column += 1
        offset += height
    return matrix


def _stack_features(prepared_dir: str, cutoff: str, numeric_rows: list,
                    categorical_rows: list) -> np.ndarray:
    """Собирает матрицу «строка x признак» из отображённых в память срезов."""
    parts = []
    if numeric_rows:
        block = _memory_map(prepared_dir, f"num_{cutoff}.npy")[numeric_rows, :]
        parts.append(np.asarray(block, dtype=np.float32).T)
    if categorical_rows:
        block = _memory_map(prepared_dir, f"cat_{cutoff}.npy")[categorical_rows, :]
        parts.append(np.asarray(block, dtype=np.float32).T)
    if not parts:
        return np.zeros((0, 0), dtype=np.float32)
    if len(parts) == 1:
        return parts[0]
    return np.hstack(parts)


def anchor_file(target_mode: str, cutoff: str) -> str:
    """Имя файла якоря: у адаптивных и снимочных таргетов он свой."""
    column = ADAPTIVE_ANCHORS.get(target_mode) or SNAPSHOT_ANCHORS.get(target_mode)
    if column:
        return f"{column}_{cutoff}.npy"
    return f"anchor_{cutoff}.npy"


def materialize(prepared_dir: str, fold: dict, feature_columns: list,
                target_mode: str, layout: str = "array") -> dict:
    """Собирает обучающую и проверочную матрицы одного фолда."""
    manifest = load_manifest(prepared_dir)
    numeric_index = {name: row for row, name in enumerate(manifest["numeric"])}
    categorical_index = {name: row for row, name in enumerate(manifest["categorical"])}

    numeric_names = [name for name in feature_columns if name in numeric_index]
    categorical_names = [name for name in feature_columns if name in categorical_index]
    numeric_rows = [numeric_index[name] for name in numeric_names]
    categorical_rows = [categorical_index[name] for name in categorical_names]

    train_targets, train_anchors, block_ids = [], [], []
    for block, cutoff in enumerate(fold["train_cutoffs"]):
        target = np.asarray(_memory_map(prepared_dir, f"y_{cutoff}.npy"))
        train_targets.append(target)
        train_anchors.append(np.asarray(
            _memory_map(prepared_dir, anchor_file(target_mode, cutoff))))
        block_ids.append(np.full(len(target), block, dtype=np.int32))

    valid_cutoff = fold["valid_cutoff"]
    columns = numeric_names + categorical_names
    if layout == "frame":
        train_features = _build_frame(prepared_dir, list(fold["train_cutoffs"]),
                                      numeric_names, numeric_rows,
                                      categorical_names, categorical_rows)
        valid_features = _build_frame(prepared_dir, [valid_cutoff], numeric_names,
                                      numeric_rows, categorical_names,
                                      categorical_rows)
    else:
        train_features = _build_matrix(prepared_dir, list(fold["train_cutoffs"]),
                                       numeric_rows, categorical_rows)
        valid_features = _build_matrix(prepared_dir, [valid_cutoff], numeric_rows,
                                       categorical_rows)
    return {
        "X_train": train_features,
        "y_train_raw": np.concatenate(train_targets),
        "anchor_train": np.concatenate(train_anchors),
        "block_ids": np.concatenate(block_ids),
        "X_valid": valid_features,
        "y_valid_raw": np.asarray(_memory_map(prepared_dir, f"y_{valid_cutoff}.npy")),
        "anchor_valid": np.asarray(
            _memory_map(prepared_dir, anchor_file(target_mode, valid_cutoff))),
        "columns": columns,
        "categorical_positions": list(range(len(numeric_names), len(columns))),
    }


def materialize_targets(prepared_dir: str, fold: dict,
                        target_mode: str = "log_level") -> dict:
    """Только таргеты и якоря фолда: признаки уже лежат проквантованными."""
    targets, anchors, block_ids = [], [], []
    for block, cutoff in enumerate(fold["train_cutoffs"]):
        target = np.asarray(_memory_map(prepared_dir, f"y_{cutoff}.npy"))
        targets.append(target)
        anchors.append(np.asarray(
            _memory_map(prepared_dir, anchor_file(target_mode, cutoff))))
        block_ids.append(np.full(len(target), block, dtype=np.int32))
    valid_cutoff = fold["valid_cutoff"]
    return {"X_train": None, "X_valid": None,
            "y_train_raw": np.concatenate(targets),
            "anchor_train": np.concatenate(anchors),
            "block_ids": np.concatenate(block_ids),
            "y_valid_raw": np.asarray(_memory_map(prepared_dir, f"y_{valid_cutoff}.npy")),
            "anchor_valid": np.asarray(
                _memory_map(prepared_dir, anchor_file(target_mode, valid_cutoff))),
            "columns": [], "categorical_positions": []}


def temporal_shuffle(target: np.ndarray, block_ids: np.ndarray, seed: int) -> np.ndarray:
    """Перемешивает таргет внутри временных блоков, а не по всей выборке."""
    generator = np.random.RandomState(seed)
    shuffled = target.copy()
    for block in np.unique(block_ids):
        positions = np.where(block_ids == block)[0]
        if len(positions) < 2:
            continue
        shuffled[positions] = target[generator.permutation(positions)]
    return shuffled


_WINDOW_TOKENS = ("7d", "14d", "30d", "60d", "90d", "180d", "365d", "lifetime",
                  "hl30", "hl60", "cat")


def feature_family(name: str) -> str:
    """Семья признака: общий префикс без хвостового токена-окна."""
    tokens = name.split("__", 1)[0].split("_")
    while tokens and (tokens[-1] in _WINDOW_TOKENS or tokens[-1].isdigit()):
        tokens.pop()
    if not tokens:
        return name
    return "_".join(tokens)

Writing nsel_iz_noutbuka/nsel/data.py


### `nsel/encoding.py` — ordered target encoding для XGBoost и LightGBM

CatBoost обрабатывает категории сам, сюда он не заходит. Для двух других семейств код
категории заменяется сглаженным средним таргета:

* статистики берутся **только из обучающей части своего фолда**;
* обучающая часть кодируется упорядоченно: строки идут блоками по времени, и каждый
  блок видит статистику только предыдущих блоков. Так строка не видит собственный таргет;
* сила сглаживания оценивается эмпирическим Байесом, методом моментов: отношение
  внутригрупповой дисперсии к межгрупповой;
* для проверочной части используется полная статистика обучения, а невиданная категория
  получает глобальное среднее.

In [6]:
%%writefile nsel_iz_noutbuka/nsel/encoding.py
"""Ordered target encoding с эмпирико-байесовским сглаживанием.

Действует только для XGBoost и LightGBM. CatBoost сюда не заходит: у него своя
обработка категорий, и трогать её запрещено.

Все статистики берутся исключительно из обучающей части своего фолда —
средние по категориям, глобальное априорное, оценка дисперсии и само
сглаживание.
"""
from __future__ import annotations

import numpy as np


def _global_mean(target: np.ndarray) -> float:
    """Априорное среднее по обучающей части фолда."""
    if len(target) == 0:
        return 0.0
    return float(np.mean(target))


def _category_statistics(codes: np.ndarray, target: np.ndarray,
                         cardinality: int) -> tuple:
    """Счётчики, суммы и суммы квадратов таргета по категориям."""
    counts = np.bincount(codes, minlength=cardinality).astype(np.float64)
    sums = np.bincount(codes, weights=target, minlength=cardinality)
    squares = np.bincount(codes, weights=target * target, minlength=cardinality)
    return counts, sums, squares


def estimate_smoothing(counts: np.ndarray, sums: np.ndarray, squares: np.ndarray,
                       prior_mean: float, config) -> float:
    """Оценивает силу сглаживания методом моментов."""
    if config.mode == "fixed":
        return float(config.fixed_smoothing)
    present = counts > 0
    if present.sum() < 2:
        return float(config.fixed_smoothing)
    means = np.zeros_like(sums)
    means[present] = sums[present] / counts[present]
    total = counts.sum()
    within = squares.sum() - (sums[present] ** 2 / counts[present]).sum()
    degrees = max(1.0, total - present.sum())
    within_variance = within / degrees
    weights = counts[present] / total
    between_variance = float(np.sum(weights * (means[present] - prior_mean) ** 2))
    harmonic = present.sum() / np.sum(1.0 / counts[present])
    between_variance = between_variance - within_variance / max(1.0, harmonic)
    if between_variance < config.minimum_variance:
        return float(config.maximum_smoothing)
    return float(min(config.maximum_smoothing, within_variance / between_variance))


def _smoothed_values(counts: np.ndarray, sums: np.ndarray, prior_mean: float,
                     smoothing: float, config) -> np.ndarray:
    """Сглаженное среднее таргета по каждой категории."""
    values = np.full(len(counts), np.nan, dtype=np.float64)
    eligible = counts >= max(1, config.min_count)
    values[eligible] = (
        (sums[eligible] + smoothing * prior_mean) / (counts[eligible] + smoothing))
    if config.unknown_value == "global":
        values[~eligible] = prior_mean
    return values


def _fill_unseen(encoded: np.ndarray, prior_mean: float, config) -> np.ndarray:
    """Подставляет значение для категорий, не встреченных в обучении."""
    if config.unknown_value == "global":
        return np.where(np.isnan(encoded), prior_mean, encoded)
    return encoded


def encode_column(train_codes: np.ndarray, train_target: np.ndarray,
                  valid_codes: np.ndarray, block_ids: np.ndarray,
                  config) -> tuple:
    """Кодирует один категориальный столбец по обучающей части фолда."""
    cardinality = int(max(train_codes.max(initial=-1), valid_codes.max(initial=-1))) + 1
    prior_mean = _global_mean(train_target)
    order = np.argsort(block_ids, kind="stable")
    blocks = np.array_split(order, max(1, config.n_blocks))

    encoded_train = np.full(len(train_codes), np.nan, dtype=np.float64)
    counts = np.zeros(cardinality, dtype=np.float64)
    sums = np.zeros(cardinality, dtype=np.float64)
    squares = np.zeros(cardinality, dtype=np.float64)

    for position, rows in enumerate(blocks):
        if len(rows) == 0:
            continue
        if position == 0:
            if config.first_block_value == "global":
                encoded_train[rows] = prior_mean
        else:
            smoothing = estimate_smoothing(counts, sums, squares, prior_mean, config)
            values = _smoothed_values(counts, sums, prior_mean, smoothing, config)
            encoded_train[rows] = _fill_unseen(values[train_codes[rows]],
                                               prior_mean, config)
        part_counts, part_sums, part_squares = _category_statistics(
            train_codes[rows], train_target[rows], cardinality)
        counts += part_counts
        sums += part_sums
        squares += part_squares

    smoothing = estimate_smoothing(counts, sums, squares, prior_mean, config)
    values = _smoothed_values(counts, sums, prior_mean, smoothing, config)
    encoded_valid = _fill_unseen(values[valid_codes], prior_mean, config)
    diagnostics = {"prior_mean": prior_mean, "smoothing": smoothing,
                   "cardinality": cardinality,
                   "known_categories": int((counts > 0).sum())}
    return encoded_train, encoded_valid, diagnostics


def encode_matrices(X_train: np.ndarray, X_valid: np.ndarray, target: np.ndarray,
                    block_ids: np.ndarray, categorical_positions: list,
                    config) -> tuple:
    """Заменяет коды категорий сглаженным средним таргета в обеих матрицах."""
    if not categorical_positions:
        return X_train, X_valid, {}
    # Столбцы заменяются на месте: копия матрицы на миллион строк и девятьсот
    # признаков стоит четыре гигабайта на процесс, а матрицы всё равно наши.
    diagnostics = {}
    for position in categorical_positions:
        encoded_train, encoded_valid, info = encode_column(
            X_train[:, position].astype(np.int64), target,
            X_valid[:, position].astype(np.int64), block_ids, config)
        X_train[:, position] = encoded_train.astype(np.float32)
        X_valid[:, position] = encoded_valid.astype(np.float32)
        diagnostics[str(position)] = info
    return X_train, X_valid, diagnostics

Writing nsel_iz_noutbuka/nsel/encoding.py


### `nsel/model.py` — щуп трёх семейств и метрики

* `build_targets` и `restore_level` переводят таргет в пространство модели и обратно в
  `log1p` уровня.
* `score_predictions` считает RMSLE целиком и отдельно по сегментам: пользователи с
  нулевым таргетом и покупавшие. Сегменты нужны защитам правила приёма.
* `_fit_catboost`, `_fit_lightgbm`, `_fit_xgboost` — одинаковые по смыслу щупы с ранней
  остановкой по проверочному фолду. После передачи данных в `Pool` или `QuantileDMatrix`
  исходные матрицы сразу освобождаются.
* Важности возвращаются для стадии 1 и для переупорядочивания кандидатов в стадии 3.

In [7]:
%%writefile nsel_iz_noutbuka/nsel/model.py
"""Обучение щупа и метрики фолда для трёх семейств моделей."""
from __future__ import annotations

import numpy as np


def _anchor_in_log(anchor: np.ndarray, target_mode: str) -> np.ndarray:
    """Якорь в логарифмическом пространстве.

    У части таргетов якорь уже логарифмический, и второй логарифм дал бы совсем
    другую величину, не подав виду.
    """
    from .config import LOG_ANCHOR_MODES

    if target_mode in LOG_ANCHOR_MODES:
        return np.clip(anchor, 0.0, None)
    return np.log1p(np.clip(anchor, 0.0, None))


def build_targets(y_raw: np.ndarray, anchor: np.ndarray, target_mode: str) -> np.ndarray:
    """Переводит сырой таргет в пространство, в котором учится модель."""
    level = np.log1p(np.clip(y_raw, 0.0, None))
    if target_mode == "log_level":
        return level
    return level - _anchor_in_log(anchor, target_mode)


def restore_level(prediction: np.ndarray, anchor: np.ndarray,
                  target_mode: str) -> np.ndarray:
    """Восстанавливает предсказание в логарифме уровня."""
    if target_mode == "log_level":
        return np.clip(prediction, 0.0, None)
    return np.clip(prediction + _anchor_in_log(anchor, target_mode), 0.0, None)


def _root_mean_square(values: np.ndarray) -> float:
    """Корень из среднего квадрата, пустой вход даёт NaN."""
    if len(values) == 0:
        return float("nan")
    return float(np.sqrt(np.mean(values ** 2)))


def score_predictions(prediction_level: np.ndarray, truth_raw: np.ndarray) -> dict:
    """RMSLE целиком и по сегментам нулевых и покупавших пользователей."""
    truth_level = np.log1p(np.clip(truth_raw, 0.0, None))
    residual = prediction_level - truth_level
    zero_mask = truth_raw <= 0
    return {"rmsle": _root_mean_square(residual),
            "rmsle_zero": _root_mean_square(residual[zero_mask]),
            "rmsle_positive": _root_mean_square(residual[~zero_mask])}


def _catboost_model(probe: dict, seed: int, threads: int, iterations: int):
    """Создаёт CatBoost-щуп."""
    from catboost import CatBoostRegressor

    return CatBoostRegressor(
        iterations=iterations, learning_rate=probe["learning_rate"],
        depth=probe["depth"], l2_leaf_reg=probe["l2_leaf_reg"],
        loss_function="RMSE", random_seed=seed, thread_count=threads,
        min_data_in_leaf=probe["min_data_in_leaf"], verbose=False,
        allow_writing_files=False)


def _fit_catboost(matrices: dict, probe: dict, seed: int, threads: int,
                  iterations: int, early_stopping: int) -> tuple:
    """Обучает CatBoost с родной обработкой категорий."""
    from catboost import Pool

    positions = matrices["categorical_positions"]
    train_pool = Pool(matrices["X_train"], matrices["y_train"], cat_features=positions)
    valid_pool = Pool(matrices["X_valid"], matrices["y_valid"], cat_features=positions)
    # Матрицы уже проквантованы внутри Pool, исходные таблицы держать незачем.
    matrices["X_train"] = None
    matrices["X_valid"] = None
    model = _catboost_model(probe, seed, threads, iterations)
    fit_arguments = {}
    if early_stopping > 0:
        fit_arguments["eval_set"] = valid_pool
        fit_arguments["use_best_model"] = True
        fit_arguments["early_stopping_rounds"] = early_stopping
    model.fit(train_pool, **fit_arguments)
    importance = np.asarray(model.get_feature_importance(train_pool), dtype=np.float64)
    return model.predict(valid_pool), importance, int(model.tree_count_)


def _fit_lightgbm(matrices: dict, probe: dict, seed: int, threads: int,
                  iterations: int, early_stopping: int) -> tuple:
    """Обучает LightGBM на матрицах с уже закодированными категориями."""
    import lightgbm as lgb

    parameters = {"objective": "regression", "metric": "rmse",
                  "learning_rate": probe["learning_rate"],
                  "max_depth": probe["depth"], "num_leaves": 2 ** probe["depth"],
                  "lambda_l2": probe["l2_leaf_reg"], "seed": seed,
                  "num_threads": threads, "verbosity": -1,
                  "bagging_fraction": probe["subsample"], "bagging_freq": 1,
                  "feature_fraction": probe["colsample"],
                  "min_data_in_leaf": probe["min_data_in_leaf"]}
    train_set = lgb.Dataset(matrices["X_train"], label=matrices["y_train"],
                            free_raw_data=True)
    callbacks = []
    valid_sets = []
    if early_stopping > 0:
        valid_sets.append(lgb.Dataset(matrices["X_valid"], label=matrices["y_valid"],
                                      reference=train_set, free_raw_data=True))
        callbacks.append(lgb.early_stopping(early_stopping, verbose=False))
    booster = lgb.train(parameters, train_set, num_boost_round=iterations,
                        valid_sets=valid_sets, callbacks=callbacks)
    importance = np.asarray(booster.feature_importance("gain"), dtype=np.float64)
    return (booster.predict(matrices["X_valid"]), importance,
            int(booster.current_iteration()))


def _fit_xgboost(matrices: dict, probe: dict, seed: int, threads: int,
                 iterations: int, early_stopping: int) -> tuple:
    """Обучает XGBoost на матрицах с уже закодированными категориями."""
    import xgboost as xgb

    parameters = {"objective": "reg:squarederror", "eval_metric": "rmse",
                  "tree_method": "hist",
                  "learning_rate": probe["learning_rate"], "max_depth": probe["depth"],
                  "reg_lambda": probe["l2_leaf_reg"], "seed": seed,
                  "nthread": threads, "subsample": probe["subsample"],
                  "colsample_bytree": probe["colsample"],
                  "min_child_weight": probe["min_data_in_leaf"]}
    # QuantileDMatrix бинует один раз при сборке и не держит исходную матрицу
    # рядом с гистограммами: это штатный способ кормить hist, и он снимает
    # лишние гигабайты на процесс.
    n_features = matrices["X_train"].shape[1]
    train_matrix = xgb.QuantileDMatrix(matrices["X_train"], label=matrices["y_train"],
                                       nthread=threads)
    valid_matrix = xgb.QuantileDMatrix(matrices["X_valid"], label=matrices["y_valid"],
                                       nthread=threads, ref=train_matrix)
    matrices["X_train"] = None
    matrices["X_valid"] = None
    arguments = {"num_boost_round": iterations, "verbose_eval": False}
    if early_stopping > 0:
        arguments["evals"] = [(valid_matrix, "valid")]
        arguments["early_stopping_rounds"] = early_stopping
    booster = xgb.train(parameters, train_matrix, **arguments)
    scores = booster.get_score(importance_type="gain")
    importance = np.array(
        [scores.get(f"f{index}", 0.0) for index in range(n_features)],
        dtype=np.float64)
    return (booster.predict(valid_matrix), importance,
            int(booster.num_boosted_rounds()))


_FITTERS = {"catboost": _fit_catboost, "lightgbm": _fit_lightgbm,
            "xgboost": _fit_xgboost}


def fit_and_score(matrices: dict, model_mode: str, target_mode: str, probe: dict,
                  seed: int, threads: int, iterations: int,
                  early_stopping: int) -> dict:
    """Обучает щуп на одном фолде и возвращает метрики с важностями."""
    prediction, importance, trees = _FITTERS[model_mode](
        matrices, probe, seed, threads, iterations, early_stopping)
    level = restore_level(prediction, matrices["anchor_valid"], target_mode)
    metrics = score_predictions(level, matrices["y_valid_raw"])
    metrics["trees"] = trees
    metrics["importance"] = importance.tolist()
    return metrics

Writing nsel_iz_noutbuka/nsel/model.py


### `nsel/worker.py` — что делает процесс-исполнитель

`run_training(job)` — единственная функция, которую пул отправляет в процессы. Одно
задание — это «набор признаков × один фолд × один сид». Шаги:

1. собрать матрицы фолда из `mmap` (`data.materialize`);
2. построить таргет; у null-модели он перемешан внутри временных блоков, проверочный тоже,
   поэтому ранняя остановка не знает настоящей цели и утечки нет;
3. закодировать категории (кроме CatBoost);
4. обучить щуп и вернуть метрики, число деревьев, время и пик памяти.

Ошибка возвращается текстом, а не исключением: исключение библиотеки модели нельзя
распаковать в родителе, где эта библиотека не импортирована.

In [8]:
%%writefile nsel_iz_noutbuka/nsel/worker.py
"""Сторона процесса-исполнителя: одно обучение по описанию задания."""
from __future__ import annotations

import os

# Потоки библиотеки ограничиваются до её импорта: иначе каждый воркер поднимет
# пул на все ядра машины и процессы начнут отбирать ядра друг у друга.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import resource
import sys
import time

import numpy as np

from . import data, encoding, model
from .config import EncodingConfig


def _apply_categoricals(matrices: dict, job: dict) -> dict:
    """Готовит категориальные столбцы под конкретное семейство моделей."""
    positions = matrices["categorical_positions"]
    if job["model_mode"] == "catboost":
        return {"X_train": matrices["X_train"], "X_valid": matrices["X_valid"],
                "categorical_positions": positions, "encoding": {}}
    config = EncodingConfig(**job["encoding"])
    train, valid, diagnostics = encoding.encode_matrices(
        matrices["X_train"], matrices["X_valid"], matrices["y_train"],
        matrices["block_ids"], positions, config)
    return {"X_train": train, "X_valid": valid, "categorical_positions": [],
            "encoding": diagnostics}


def peak_memory_mb() -> float:
    """Пик памяти процесса-исполнителя за всё его время жизни."""
    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    if sys.platform == "darwin":
        return peak / 1e6
    return peak / 1024.0


def _layout_for(model_mode: str) -> str:
    """CatBoost получает таблицу с целыми категориями, остальные — матрицу."""
    if model_mode == "catboost":
        return "frame"
    return "array"


def _training_target(matrices: dict, job: dict) -> np.ndarray:
    """Таргет обучения; для null-модели он перемешан внутри временных блоков."""
    target = model.build_targets(matrices["y_train_raw"], matrices["anchor_train"],
                                 job["target_mode"])
    if job.get("shuffle_seed") is None:
        return target
    return data.temporal_shuffle(target, matrices["block_ids"], job["shuffle_seed"])


def _validation_target(matrices: dict, job: dict) -> np.ndarray:
    """Проверочный таргет; у null-модели он перемешан тоже.

    Ранняя остановка тогда смотрит на разрушенную цель и о настоящей не знает
    ничего — утечки нет, а обучение обрывается быстро, потому что улучшать
    нечего.
    """
    target = model.build_targets(matrices["y_valid_raw"], matrices["anchor_valid"],
                                 job["target_mode"])
    if job.get("shuffle_seed") is None:
        return target
    blocks = np.zeros(len(target), dtype=np.int32)
    return data.temporal_shuffle(target, blocks, job["shuffle_seed"] + 1)


class TrainingFailed(RuntimeError):
    """Обучение в процессе-исполнителе не удалось; текст пришёл из воркера."""


def run_training(job: dict) -> dict:
    """Обучает щуп на одном фолде, возвращая ошибку текстом, а не объектом.

    Исключения библиотек моделей распаковываются только там, где библиотека
    импортирована. Родитель импортирует её лениво и у него её нет, поэтому
    настоящая ошибка воркера превращалась в невнятное «процесс убит».
    """
    import traceback

    try:
        return _train(job)
    except BaseException as error:
        return {"error": f"{type(error).__name__}: {error}",
                "traceback": traceback.format_exc()[-4000:]}


def _train(job: dict) -> dict:
    """Обучает щуп на одном фолде и возвращает метрики."""
    started = time.perf_counter()
    matrices = data.materialize(job["prepared_dir"], job["fold"],
                                job["feature_columns"], job["target_mode"],
                                layout=_layout_for(job["model_mode"]))
    matrices["y_train"] = _training_target(matrices, job)
    matrices["y_valid"] = _validation_target(matrices, job)

    prepared = _apply_categoricals(matrices, job)
    payload = dict(matrices)
    payload.update(prepared)
    # Второй словарь держал бы вторую ссылку на матрицы, и освободить их до
    # конца обучения было бы нечем.
    matrices.clear()
    prepared_encoding = prepared["encoding"]
    prepared.clear()

    metrics = model.fit_and_score(
        payload, job["model_mode"], job["target_mode"], job["probe"], job["seed"],
        job["threads"], job["iterations"], job["early_stopping"])
    metrics["fold"] = job["fold"]["name"]
    metrics["weight"] = job["fold"]["weight"]
    metrics["n_features"] = len(job["feature_columns"])
    metrics["seconds"] = time.perf_counter() - started
    metrics["peak_rss_mb"] = peak_memory_mb()
    metrics["encoding"] = prepared_encoding
    if not job.get("want_importance", False):
        metrics.pop("importance", None)
    return metrics

Writing nsel_iz_noutbuka/nsel/worker.py


### `nsel/jobs.py` — нарезка на задания и сведение метрик

* `build_jobs` строит по заданию на каждый фолд. Ключ кэша — sha256 от всего, что влияет
  на результат: отпечатка прогона, фолда, отсортированного набора признаков, модели,
  таргета, параметров щупа, сида, перемешивания, кодирования и числа потоков.
* `aggregate` сводит метрики фолдов во взвешенную метрику набора.
* `improvement_profile` считает выигрыш кандидата относительно текущего набора: взвешенный,
  по фолдам, число улучшенных фолдов и худшие просадки по фолду и сегментам.

In [9]:
%%writefile nsel_iz_noutbuka/nsel/jobs.py
"""Сборка заданий обучения и взвешивание метрик по фолдам."""
from __future__ import annotations

import dataclasses

from . import cache


def build_probe(config) -> dict:
    """Параметры щупа в виде словаря для передачи в воркер."""
    return dataclasses.asdict(config.probe)


def build_jobs(context: dict, feature_columns: list, seed: int, iterations: int,
               early_stopping: int, shuffle_seed=None,
               want_importance: bool = False, folds=None, wide: bool = False) -> list:
    """Задания «обучить набор на каждом фолде» с ключами кэша."""
    config = context["config"]
    folds = folds or context["folds"]
    # Повтор имени столбца недопустим ни для одной библиотеки и всегда означает
    # ошибку вызывающего, а не намерение: набор признаков — это множество.
    columns = sorted(set(feature_columns))
    encoding = dataclasses.asdict(config.encoding)
    probe = build_probe(config)
    threads = config.parallel.threads_for(wide)
    jobs = []
    for fold in folds:
        job = {"prepared_dir": str(config.prepared_dir), "fold": fold.as_dict(),
               "feature_columns": columns, "model_mode": config.model_mode,
               "target_mode": config.target_mode, "probe": probe, "seed": seed,
               "threads": threads,
               "iterations": iterations, "early_stopping": early_stopping,
               "shuffle_seed": shuffle_seed, "encoding": encoding,
               "want_importance": want_importance}
        key = cache.make_key(context["fingerprint"], fold.key_parts(), columns,
                             config.model_mode, config.target_mode, probe, seed,
                             iterations, early_stopping, shuffle_seed, encoding,
                             want_importance, threads)
        jobs.append((key, job))
    return jobs


def aggregate(results: dict, jobs: list, folds: list) -> dict:
    """Взвешенная метрика набора и разбивка по фолдам."""
    by_fold = {}
    for key, job in jobs:
        payload = results.get(key)
        if payload is None:
            raise KeyError(f"нет результата для фолда {job['fold']['name']}")
        by_fold[job["fold"]["name"]] = payload
    weighted = 0.0
    for fold in folds:
        weighted += fold.weight * by_fold[fold.name]["rmsle"]
    return {"weighted": weighted,
            "per_fold": {name: {"rmsle": payload["rmsle"],
                                "rmsle_zero": payload["rmsle_zero"],
                                "rmsle_positive": payload["rmsle_positive"],
                                "trees": payload["trees"]}
                         for name, payload in by_fold.items()}}


def improvement_profile(baseline: dict, candidate: dict, folds: list) -> dict:
    """Выигрыш набора относительно базы: взвешенный и по каждому фолду."""
    per_fold = {}
    improved = 0
    worst_degradation = 0.0
    worst_zero = 0.0
    worst_positive = 0.0
    for fold in folds:
        base = baseline["per_fold"][fold.name]
        new = candidate["per_fold"][fold.name]
        delta = base["rmsle"] - new["rmsle"]
        per_fold[fold.name] = delta
        if delta > 0:
            improved += 1
        worst_degradation = max(worst_degradation, -delta)
        worst_zero = max(worst_zero, new["rmsle_zero"] - base["rmsle_zero"])
        worst_positive = max(worst_positive,
                             new["rmsle_positive"] - base["rmsle_positive"])
    return {"weighted_improvement": baseline["weighted"] - candidate["weighted"],
            "per_fold_improvement": per_fold, "improved_folds": improved,
            "worst_fold_degradation": worst_degradation,
            "worst_zero_degradation": worst_zero,
            "worst_positive_degradation": worst_positive}

Writing nsel_iz_noutbuka/nsel/jobs.py


### `nsel/cache.py` — кэш в sqlite

Три таблицы:

| таблица | что хранит | зачем |
|---|---|---|
| `iterations` | результат каждого обучения по ключу | одно и то же обучение никогда не считается дважды |
| `stage_state` | состояние стадии после каждого решения | продолжение с середины стадии |
| `stage_results` | итог стадии вместе с отпечатком | готовая стадия при повторном запуске пропускается |

Журнал WAL и повтор при блокировке нужны потому, что в базу пишут результаты из всех
процессов пула.

In [10]:
%%writefile nsel_iz_noutbuka/nsel/cache.py
"""Кэш прогона: итерации, промежуточное состояние стадии и итоги стадий."""
from __future__ import annotations

import hashlib
import json
import sqlite3
import time
from pathlib import Path

SCHEMA = """
CREATE TABLE IF NOT EXISTS iterations (
    key TEXT PRIMARY KEY, stage TEXT NOT NULL,
    payload TEXT NOT NULL, created_at REAL NOT NULL);
CREATE INDEX IF NOT EXISTS iterations_stage ON iterations(stage);
CREATE TABLE IF NOT EXISTS stage_state (
    stage TEXT NOT NULL, checkpoint TEXT NOT NULL, payload TEXT NOT NULL,
    updated_at REAL NOT NULL, PRIMARY KEY (stage, checkpoint));
CREATE TABLE IF NOT EXISTS stage_results (
    stage TEXT PRIMARY KEY, fingerprint TEXT NOT NULL,
    payload TEXT NOT NULL, finished_at REAL NOT NULL);
"""


def make_key(*parts) -> str:
    """Ключ кэша по всем параметрам, влияющим на результат."""
    blob = json.dumps(parts, sort_keys=True, default=str,
                      ensure_ascii=False).encode("utf-8")
    return hashlib.sha256(blob).hexdigest()


def _connect(db_path) -> sqlite3.Connection:
    """Открывает короткоживущее соединение с журналом WAL."""
    Path(db_path).parent.mkdir(parents=True, exist_ok=True)
    connection = sqlite3.connect(str(db_path), timeout=60.0, isolation_level=None)
    connection.execute("PRAGMA journal_mode=WAL")
    connection.execute("PRAGMA synchronous=NORMAL")
    connection.execute("PRAGMA busy_timeout=60000")
    return connection


def _with_retry(action, retries: int = 12, base_delay: float = 0.05):
    """Повторяет операцию при блокировке базы."""
    last = None
    for attempt in range(retries):
        try:
            return action()
        except sqlite3.OperationalError as error:
            last = error
            if "lock" not in str(error).lower() or attempt == retries - 1:
                raise
            time.sleep(base_delay * (2 ** attempt))
    raise last


def init_db(db_path) -> None:
    """Создаёт таблицы кэша, если их ещё нет."""
    connection = _connect(db_path)
    try:
        connection.executescript(SCHEMA)
    finally:
        connection.close()


def _query(db_path, sql: str, parameters: tuple):
    """Выполняет чтение и возвращает первую строку."""
    def action():
        connection = _connect(db_path)
        try:
            return connection.execute(sql, parameters).fetchone()
        finally:
            connection.close()

    return _with_retry(action)


def _execute(db_path, sql: str, parameters: tuple) -> None:
    """Выполняет запись."""
    def action():
        connection = _connect(db_path)
        try:
            connection.execute(sql, parameters)
        finally:
            connection.close()

    _with_retry(action)


def get_iteration(db_path, key: str):
    """Результат одного обучения из кэша или None."""
    row = _query(db_path, "SELECT payload FROM iterations WHERE key = ?", (key,))
    if row is None:
        return None
    return json.loads(row[0])


def get_iterations(db_path, keys: list) -> dict:
    """Пакетное чтение результатов обучений по списку ключей."""
    if not keys:
        return {}

    def action():
        connection = _connect(db_path)
        try:
            found = {}
            for start in range(0, len(keys), 500):
                chunk = keys[start:start + 500]
                marks = ",".join("?" * len(chunk))
                rows = connection.execute(
                    f"SELECT key, payload FROM iterations WHERE key IN ({marks})",
                    tuple(chunk)).fetchall()
                found.update({key: json.loads(payload) for key, payload in rows})
            return found
        finally:
            connection.close()

    return _with_retry(action)


def put_iteration(db_path, key: str, stage: str, payload: dict) -> None:
    """Сохраняет результат обучения сразу по его готовности."""
    _execute(db_path,
             "INSERT OR REPLACE INTO iterations (key, stage, payload, created_at) "
             "VALUES (?, ?, ?, ?)",
             (key, stage, json.dumps(payload, ensure_ascii=False), time.time()))


def put_stage_state(db_path, stage: str, checkpoint: str, payload: dict) -> None:
    """Сохраняет промежуточное состояние стадии для возобновления."""
    _execute(db_path,
             "INSERT OR REPLACE INTO stage_state (stage, checkpoint, payload, "
             "updated_at) VALUES (?, ?, ?, ?)",
             (stage, checkpoint, json.dumps(payload, ensure_ascii=False), time.time()))


def get_stage_state(db_path, stage: str, checkpoint: str):
    """Промежуточное состояние стадии или None."""
    row = _query(db_path,
                 "SELECT payload FROM stage_state WHERE stage = ? AND checkpoint = ?",
                 (stage, checkpoint))
    if row is None:
        return None
    return json.loads(row[0])


def latest_stage_state(db_path, stage: str):
    """Самое свежее промежуточное состояние стадии или None."""
    row = _query(db_path,
                 "SELECT payload FROM stage_state WHERE stage = ? "
                 "ORDER BY updated_at DESC LIMIT 1", (stage,))
    if row is None:
        return None
    return json.loads(row[0])


def put_stage_result(db_path, stage: str, fingerprint: str, payload: dict) -> None:
    """Записывает итог стадии."""
    _execute(db_path,
             "INSERT OR REPLACE INTO stage_results (stage, fingerprint, payload, "
             "finished_at) VALUES (?, ?, ?, ?)",
             (stage, fingerprint, json.dumps(payload, ensure_ascii=False), time.time()))


def get_stage_result(db_path, stage: str, fingerprint: str):
    """Итог стадии, если он посчитан с тем же отпечатком настроек."""
    row = _query(db_path,
                 "SELECT payload, fingerprint FROM stage_results WHERE stage = ?",
                 (stage,))
    if row is None:
        return None
    payload, saved = row
    if saved != fingerprint:
        return None
    return json.loads(payload)


def count_iterations(db_path, stage: str) -> int:
    """Сколько обучений стадии лежит в кэше."""
    row = _query(db_path, "SELECT COUNT(*) FROM iterations WHERE stage = ?", (stage,))
    return int(row[0])

Writing nsel_iz_noutbuka/nsel/cache.py


### `nsel/pool.py` — пул процессов, батчи и счёт на опережение

* `make_pool` поднимает `ProcessPoolExecutor` способом `spawn`. `_worker_init` задаёт
  процессу `OMP_NUM_THREADS` до импорта библиотек (иначе каждый процесс займёт все ядра)
  и запускает сторожа, который гасит процесс, если родитель умер.
* `run_jobs_cached` — **батч**. Берёт из кэша всё, что уже посчитано, остальное
  отправляет в пул одной пачкой и пишет каждый результат в кэш сразу по готовности.
* `iterate_in_order` — **счёт на опережение**. Отдаёт результаты кандидатов строго в
  порядке ранга, но заранее отправляет в пул следующих. Стартовая ширина берётся из
  медианы длины прошлых раундов и удваивается на каждом отказе, до потолка. Когда
  кандидат принят, вызывающий закрывает генератор, и лишние задания отменяются.
  Отвергнутый кандидат не меняет состояние, поэтому решения совпадают с
  последовательным проходом.
* `WorkerLost` переводит гибель процесса (обычно от нехватки памяти) в понятный текст.
  Оркестратор в этом случае поднимает пул заново и повторяет стадию, и посчитанное
  берётся из кэша.

In [11]:
%%writefile nsel_iz_noutbuka/nsel/pool.py
"""Пул процессов: непрерывная очередь заданий и упорядоченная спекуляция."""
from __future__ import annotations

import logging
import multiprocessing
import os
import sys
import threading
import time
from concurrent.futures import (
    FIRST_COMPLETED,
    BrokenExecutor,
    ProcessPoolExecutor,
    wait,
)

import psutil

from . import cache
from .worker import TrainingFailed, run_training

POLL_INTERVAL = 0.5
WATCHDOG_INTERVAL = 1.0


class StopRequested(Exception):
    """Пришёл сигнал остановки посреди стадии."""


class WorkerLost(RuntimeError):
    """Процесс-исполнитель убит извне, чаще всего нехваткой памяти."""


def _check_payload(payload: dict, stage: str) -> dict:
    """Поднимает ошибку воркера, пришедшую текстом."""
    if "error" in payload:
        raise TrainingFailed(f"[{stage}] {payload['error']}\n{payload['traceback']}")
    return payload


def _result_or_explain(future, stage: str):
    """Достаёт результат задания, объясняя гибель процесса понятным текстом."""
    try:
        return _check_payload(future.result(), stage)
    except BrokenExecutor as error:
        raise WorkerLost(
            f"[{stage}] процесс-исполнитель убит: {error}. Обычная причина — "
            "нехватка памяти: уменьшите число процессов или поднимите "
            "--memory-per-worker") from error


def _worker_init(parent_pid: int, parent_created: float, threads: int) -> None:
    """Задаёт воркеру число потоков и не даёт ему пережить родителя.

    XGBoost и LightGBM параллелятся через OpenMP, и `OMP_NUM_THREADS` перебивает
    их собственные параметры: замер показал, что при значении 1 аргумент
    nthread=10 не даёт ничего, а без ограничения то же обучение идёт в 6.4 раза
    быстрее. Значение ставится здесь, до первого импорта библиотеки в этом
    процессе, и равно числу потоков, выделенных одному обучению.
    """
    os.environ["OMP_NUM_THREADS"] = str(max(1, threads))
    os.environ["MKL_NUM_THREADS"] = str(max(1, threads))
    if sys.platform.startswith("linux"):
        try:
            import ctypes

            ctypes.CDLL("libc.so.6", use_errno=True).prctl(1, 9, 0, 0, 0)
        except Exception:
            pass

    def watch() -> None:
        while True:
            time.sleep(WATCHDOG_INTERVAL)
            try:
                parent = psutil.Process(parent_pid)
                if parent.create_time() != parent_created or not parent.is_running():
                    os._exit(1)
            except psutil.NoSuchProcess:
                os._exit(1)

    threading.Thread(target=watch, daemon=True).start()


def make_pool(n_workers: int, threads: int = 1) -> ProcessPoolExecutor:
    """Поднимает пул процессов способом spawn."""
    parent = psutil.Process(os.getpid())
    return ProcessPoolExecutor(
        max_workers=n_workers, mp_context=multiprocessing.get_context("spawn"),
        initializer=_worker_init,
        initargs=(parent.pid, parent.create_time(), threads))


def _child_memory(child) -> int:
    """Память дочернего процесса; исчезнувший и зомби считаются нулём."""
    try:
        return child.memory_info().rss
    except (psutil.NoSuchProcess, psutil.ZombieProcess, psutil.AccessDenied):
        return 0


def memory_summary() -> str:
    """Короткая сводка по памяти процесса и его детей."""
    process = psutil.Process()
    own = process.memory_info().rss
    children = sum(_child_memory(child) for child in process.children(recursive=True))
    return (f"память={own / 1e6:.0f}МБ дети={children / 1e6:.0f}МБ "
            f"занято={psutil.virtual_memory().percent:.0f}%")


def _split_cached(db_path, jobs: list) -> tuple:
    """Делит задания на посчитанные ранее и требующие счёта."""
    known = cache.get_iterations(db_path, [key for key, _ in jobs])
    pending = [(key, job) for key, job in jobs if key not in known]
    return known, pending


def run_jobs_cached(pool, db_path, stage: str, jobs: list, logger: logging.Logger,
                    stop_check) -> dict:
    """Считает все задания стадии, сохраняя каждый результат сразу."""
    results, pending = _split_cached(db_path, jobs)
    logger.info("[%s] заданий %d, из кэша %d, к счёту %d",
                stage, len(jobs), len(results), len(pending))
    if not pending:
        return results

    futures = {pool.submit(run_training, job): key for key, job in pending}
    waiting = set(futures)
    started = time.time()
    done_count = 0
    try:
        while waiting:
            if stop_check():
                raise StopRequested(stage)
            finished, waiting = wait(waiting, timeout=POLL_INTERVAL,
                                     return_when=FIRST_COMPLETED)
            for future in finished:
                key = futures[future]
                payload = _result_or_explain(future, stage)
                cache.put_iteration(db_path, key, stage, payload)
                results[key] = payload
                done_count += 1
                elapsed = time.time() - started
                remaining = elapsed / done_count * (len(pending) - done_count)
                logger.info("[%s] %d/%d прошло %.0fс осталось ~%.0fс пик %.0fМБ %s",
                            stage, done_count, len(pending), elapsed, remaining,
                            payload.get("peak_rss_mb", 0.0), memory_summary())
    finally:
        for future in waiting:
            future.cancel()
    return results


def iterate_in_order(pool, db_path, stage: str, units: list, logger: logging.Logger,
                     stop_check, width: int, start_width: int = 1):
    """Выдаёт результаты единиц работы строго по рангу, считая их с опережением.

    Единица — список заданий одного кандидата. Опережающий счёт не меняет
    решений: отвергнутый кандидат не меняет состояние отбора, а на принятом
    очередь обрывается ровно там же, где оборвался бы последовательный проход.

    Ширина начинается с предсказанной по прошлым раундам и удваивается внутри
    раунда на каждом отвергнутом кандидате, до потолка. Так покрыты оба случая:
    короткий раунд, который кончится приёмом на первом же кандидате, ничего не
    выбрасывает, а длинная полоса отказов быстро занимает весь пул.
    """
    in_flight = {}
    submitted = 0
    position = 0
    current = max(1, min(width, start_width))
    try:
        while position < len(units):
            while submitted < len(units) and len(in_flight) < current:
                name, jobs = units[submitted]
                known, pending = _split_cached(db_path, jobs)
                futures = {pool.submit(run_training, job): key for key, job in pending}
                in_flight[submitted] = (name, jobs, known, futures)
                submitted += 1
            name, jobs, known, futures = in_flight.pop(position)
            for future in list(futures):
                if stop_check():
                    raise StopRequested(stage)
                key = futures[future]
                payload = _result_or_explain(future, stage)
                cache.put_iteration(db_path, key, stage, payload)
                known[key] = payload
            logger.debug("[%s] готова единица %s (в полёте %d)",
                         stage, name, len(in_flight))
            yield name, jobs, known
            # Управление вернулось — кандидат отвергнут, состояние не менялось,
            # значит следующие можно считать смелее.
            position += 1
            current = min(width, current * 2)
    finally:
        for _name, _jobs, _known, futures in in_flight.values():
            for future in futures:
                future.cancel()

Writing nsel_iz_noutbuka/nsel/pool.py


### `nsel/checks.py` — обязательные проверки итога

Перед записью отчёта проверяется: контракт трёх фолдов, отсутствие Group LOFO, ровно одна
обратная чистка, проверка набора на всех заданных сидах, состояние парного спасения
совпадает с настройкой, итоговый набор непустой и без повторов.

In [12]:
%%writefile nsel_iz_noutbuka/nsel/checks.py
"""Обязательные проверки итога прогона."""
from __future__ import annotations

from .config import ContractError, assert_fold_contract

FORBIDDEN_STAGES = ("group_lofo", "stage2_group_lofo", "lofo")


def check_no_group_lofo(stages_run: list) -> None:
    """Group LOFO вырезан: стадии с таким именем в прогоне быть не может."""
    for name in stages_run:
        for forbidden in FORBIDDEN_STAGES:
            if forbidden in name:
                raise ContractError(f"в прогоне встретилась стадия LOFO: {name}")


def check_single_backward(stages_run: list) -> None:
    """Обратная чистка проходит ровно один раз."""
    count = sum(1 for name in stages_run if name.startswith("stage6_backward"))
    if count != 1:
        raise ContractError(f"обратных чисток {count}, должна быть одна")


def check_multiseed(report: dict, config) -> None:
    """Сиды проверяют набор целиком, а не каждый признак по отдельности."""
    if not config.stage4.enabled:
        return
    seeds = set(report["multiseed"]["per_seed"])
    if seeds != {str(seed) for seed in config.stage4.seeds}:
        raise ContractError(f"проверка на сидах {sorted(seeds)}")


def check_pair_rescue(report: dict, config) -> None:
    """Состояние парного спасения записано и совпадает с настройкой."""
    if report["pair_rescue"]["enabled"] != config.stage5.enabled:
        raise ContractError("состояние парного спасения не совпадает с настройкой")


def check_selection(report: dict) -> None:
    """Итоговый набор непустой и без повторов.

    Повтор не меняет метрику, но ломает обучение у того, кто возьмёт список
    как есть: CatBoost отказывается работать с одинаковыми именами признаков.
    """
    features = report["selected_features"]
    if not features:
        raise ContractError("отбор вернул пустой набор")
    if len(features) != len(set(features)):
        seen, repeated = set(), []
        for name in features:
            if name in seen:
                repeated.append(name)
            seen.add(name)
        raise ContractError(f"в наборе повторяются признаки: {sorted(set(repeated))}")


def run_all(report: dict, config, folds: list, stages_run: list) -> list:
    """Прогоняет все обязательные проверки и возвращает их список."""
    assert_fold_contract(folds)
    check_no_group_lofo(stages_run)
    check_single_backward(stages_run)
    check_multiseed(report, config)
    check_pair_rescue(report, config)
    check_selection(report)
    return ["контракт трёх фолдов", "Group LOFO отсутствует",
            "одна обратная чистка", "multiseed на наборе",
            "состояние парного спасения", "набор непустой и без повторов"]

Writing nsel_iz_noutbuka/nsel/checks.py


### Стадии отбора

`nsel/stages/__init__.py` пустой.

In [13]:
Path("nsel_iz_noutbuka/nsel/stages/__init__.py").write_text("")  # файл пустой, как в оригинале

0

### Стадия 1 — временная null importance

1. `actual_runs` обучений на настоящем таргете и `null_runs` обучений на таргете,
   перемешанном внутри временных блоков. Все на полном наборе признаков, по всем фолдам.
2. Важности каждого обучения нормируются к сумме 1.
3. Score признака — взвешенное по фолдам среднее важности на настоящем таргете.
   Порог — взвешенный по фолдам квантиль 0.90 важности у null-моделей.
4. Проходят признаки выше порога **плюс** верхние `max(min_keep, 15%)` по рангу. Это
   объединение, а не замена одного другим: признак, чуть не дотянувший до порога, но
   стоящий высоко по важности, остаётся кандидатом. Сверху ограничение `max_keep`.

Задания сортируются по фолду, чтобы подряд шли обучения на одной и той же матрице.

In [14]:
%%writefile nsel_iz_noutbuka/nsel/stages/stage1_null_importance.py
"""Стадия 1: временная null importance и ранжирование признаков."""
from __future__ import annotations

import numpy as np

from .. import cache, jobs, pool
from ..logging_setup import emit_event

STAGE = "stage1_null_importance"


def _normalise(values: np.ndarray) -> np.ndarray:
    """Приводит важности одного обучения к сумме единица."""
    total = float(np.sum(np.abs(values)))
    if total <= 0:
        return np.zeros_like(values)
    return np.abs(values) / total


def _build_all_jobs(context: dict, features: list) -> tuple:
    """Задания настоящих и null-моделей для всех фолдов."""
    config = context["config"].stage1
    seed = context["config"].random_state
    actual, null = [], []
    for run in range(config.actual_runs):
        actual.extend(jobs.build_jobs(
            context, features, seed=seed + run, iterations=config.actual_iterations,
            early_stopping=config.actual_early_stopping, want_importance=True,
            wide=True))
    for run in range(config.null_runs):
        null.extend(jobs.build_jobs(
            context, features, seed=seed + 1000 + run,
            iterations=config.null_iterations,
            early_stopping=config.null_early_stopping,
            shuffle_seed=seed + 5000 + run, want_importance=True, wide=True))
    return actual, null


def order_by_fold(job_list: list) -> list:
    """Ставит подряд задания одного фолда, чтобы матрица квантовалась однажды."""
    return sorted(job_list, key=lambda item: item[1]["fold"]["name"])


def _collect(results: dict, job_list: list, folds: list, n_features: int) -> dict:
    """Собирает важности по фолдам: массив «прогон x признак» на каждый фолд."""
    per_fold = {fold.name: [] for fold in folds}
    for key, job in job_list:
        payload = results[key]
        importance = np.asarray(payload["importance"], dtype=np.float64)
        if len(importance) != n_features:
            raise ValueError(f"важностей {len(importance)}, признаков {n_features}")
        per_fold[job["fold"]["name"]].append(_normalise(importance))
    return {name: np.vstack(rows) for name, rows in per_fold.items()}


def _weighted_scores(actual: dict, null: dict, folds: list, quantile: float) -> tuple:
    """Взвешенные по фолдам важность признака и планка null-моделей."""
    score = None
    threshold = None
    for fold in folds:
        actual_mean = actual[fold.name].mean(axis=0) * fold.weight
        null_level = np.quantile(null[fold.name], quantile, axis=0) * fold.weight
        if score is None:
            score = actual_mean
            threshold = null_level
        else:
            score = score + actual_mean
            threshold = threshold + null_level
    return score, threshold


def _unique_keep_order(names: list) -> list:
    """Убирает повторы, сохраняя порядок первого появления."""
    seen = set()
    result = []
    for name in names:
        if name not in seen:
            seen.add(name)
            result.append(name)
    return result


def select_features(features: list, score: np.ndarray, threshold: np.ndarray,
                    config) -> tuple:
    """Отбирает признаки по планке null-моделей, как в его keep_mode.

    Правило у него — объединение прошедших планку с верхней долей ранжирования,
    а не замена одного другим: признак, чуть не дотянувший до планки, но стоящий
    в первой сотне по важности, остаётся кандидатом.
    """
    order = np.argsort(-score)
    ranking = [features[i] for i in order]
    passed = [features[i] for i in order if score[i] > threshold[i]]
    quantile_count = len(passed)
    top_count = max(int(config.min_keep),
                    int(np.ceil(len(features) * config.fallback_top_fraction)))
    passed = _unique_keep_order(passed + ranking[:top_count])
    if len(passed) < config.min_keep:
        passed = _unique_keep_order(passed + ranking[:config.min_keep])
    reason = "quantile_or_top_fraction"
    if quantile_count >= len(passed):
        reason = "quantile"
    if len(passed) > config.max_keep:
        passed = passed[:config.max_keep]
        reason = reason + "+max_keep"
    return passed, ranking, reason


def run(context: dict, features: list, executor, stop_check) -> dict:
    """Считает стадию 1 и возвращает прошедшие признаки в порядке важности."""
    logger = context["logger"]
    config = context["config"].stage1
    cached = cache.get_stage_result(context["cache_db"], STAGE, context["fingerprint"])
    if cached is not None:
        logger.info("[%s] итог стадии уже посчитан: прошло %d признаков",
                    STAGE, len(cached["passed"]))
        return cached
    if not config.enabled:
        return {"passed": list(features), "ranking": list(features),
                "reason": "disabled", "scores": {}}

    actual_jobs, null_jobs = _build_all_jobs(context, features)
    logger.info("[%s] настоящих обучений %d, null-обучений %d, признаков %d",
                STAGE, len(actual_jobs), len(null_jobs), len(features))
    results = pool.run_jobs_cached(executor, context["cache_db"], STAGE,
                                   order_by_fold(actual_jobs + null_jobs), logger,
                                   stop_check)

    folds = context["folds"]
    actual = _collect(results, actual_jobs, folds, len(features))
    null = _collect(results, null_jobs, folds, len(features))
    score, threshold = _weighted_scores(actual, null, folds, config.quantile)
    passed, ranking, reason = select_features(features, score, threshold, config)

    payload = {"passed": passed, "ranking": ranking, "reason": reason,
               "scores": {name: float(score[i]) for i, name in enumerate(features)},
               "thresholds": {name: float(threshold[i])
                              for i, name in enumerate(features)}}
    cache.put_stage_result(context["cache_db"], STAGE, context["fingerprint"], payload)
    logger.info("[%s] прошло %d из %d признаков, правило %s",
                STAGE, len(passed), len(features), reason)
    emit_event("stage_done", stage=STAGE, passed=len(passed), reason=reason)
    return payload

Writing nsel_iz_noutbuka/nsel/stages/stage1_null_importance.py


### Стадия 3 — ranked forward selection

* Затравка — первый по рангу признак, база — его метрика.
* Кандидаты проверяются по рангу. Каждый — это набор плюс кандидат на всех трёх фолдах.
* `decide`: сначала порог (обычный или устойчивый путь), потом защиты. Порог проверяется
  раньше защиты, чтобы кандидат ниже порога не попал в отчёт как «нарушил защиту»: иначе
  его потеряло бы парное спасение.
* Принятый кандидат меняет базу, и раунд начинается заново с оставшихся. Отвергнутый
  выбывает навсегда, поэтому объём счёта растёт линейно, а не квадратично.
* Каждые `rerank_after_n_added` добавлений остаток переупорядочивается по важности при
  текущем наборе.
* Состояние сохраняется после каждого решения (`_save_state`), так что обрыв на середине
  продолжается с того же кандидата.
* `_speculation_width` выбирает потолок и стартовую ширину опережения для `iterate_in_order`.

In [15]:
%%writefile nsel_iz_noutbuka/nsel/stages/stage3_ranked_forward.py
"""Стадия 3: ranked forward selection с двухуровневым правилом приёма."""
from __future__ import annotations

import statistics

import numpy as np

from .. import cache, jobs, pool
from ..logging_setup import emit_event

STAGE = "stage3_ranked_forward"
CHECKPOINT = "progress"


def guards_pass(profile: dict, config) -> tuple:
    """Проверяет защиту по просадке фолда и сегментов."""
    if profile["worst_fold_degradation"] > config.max_allowed_fold_degradation:
        return False, "просадка фолда"
    if profile["worst_zero_degradation"] > config.max_zero_segment_degradation:
        return False, "просадка сегмента нулей"
    if profile["worst_positive_degradation"] > config.max_positive_segment_degradation:
        return False, "просадка сегмента покупавших"
    return True, ""


def decide(profile: dict, config) -> dict:
    """Решение о приёме кандидата по обоим путям правила."""
    standard = (profile["weighted_improvement"] >= config.min_improvement
                and profile["improved_folds"] >= config.min_improved_folds)
    stable = (profile["weighted_improvement"] >= config.stable_min_improvement
              and profile["improved_folds"] >= config.stable_min_improved_folds)
    # Порог проверяется раньше защиты: иначе кандидат, не дотянувший до порога,
    # попадал бы в отчёт как «нарушил защиту» и парное спасение его теряло.
    if not (standard or stable):
        return {"accepted": False, "path": "none", "reason": "выигрыш ниже порога"}
    allowed, guard_reason = guards_pass(profile, config)
    if not allowed:
        return {"accepted": False, "path": "none", "reason": guard_reason}
    if standard:
        return {"accepted": True, "path": "standard", "reason": "обычный путь"}
    return {"accepted": True, "path": "stable",
            "reason": "устойчивый малый выигрыш"}


def _evaluate_baseline(context: dict, selected: list, executor, stop_check) -> dict:
    """Метрика текущего набора на всех фолдах."""
    config = context["config"]
    job_list = jobs.build_jobs(context, selected, seed=config.random_state,
                               iterations=config.probe.iterations,
                               early_stopping=config.probe.early_stopping_rounds)
    results = pool.run_jobs_cached(executor, context["cache_db"], STAGE, job_list,
                                   context["logger"], stop_check)
    return jobs.aggregate(results, job_list, context["folds"])


def _candidate_units(context: dict, selected: list, candidates: list) -> list:
    """Единицы работы «набор плюс кандидат» в порядке ранга."""
    config = context["config"]
    units = []
    for name in candidates:
        units.append((name, jobs.build_jobs(
            context, selected + [name], seed=config.random_state,
            iterations=config.probe.iterations,
            early_stopping=config.probe.early_stopping_rounds)))
    return units


def _rerank(context: dict, selected: list, candidates: list, executor,
            stop_check) -> list:
    """Переупорядочивает остаток кандидатов по важности при текущем наборе."""
    if not candidates:
        return candidates
    config = context["config"]
    job_list = jobs.build_jobs(context, selected + candidates,
                               seed=config.random_state,
                               iterations=config.probe.iterations,
                               early_stopping=config.probe.early_stopping_rounds,
                               want_importance=True)
    results = pool.run_jobs_cached(executor, context["cache_db"], STAGE, job_list,
                                   context["logger"], stop_check)
    columns = sorted(selected + candidates)
    score = np.zeros(len(columns))
    for key, job in job_list:
        weight = job["fold"]["weight"]
        values = np.asarray(results[key]["importance"], dtype=np.float64)
        total = float(np.sum(np.abs(values)))
        if total > 0:
            score += weight * np.abs(values) / total
    ranking = {name: score[i] for i, name in enumerate(columns)}
    return sorted(candidates, key=lambda name: -ranking[name])


def _load_state(context: dict, candidates: list, baseline: dict, seed: str) -> dict:
    """Состояние стадии из кэша или начальное."""
    saved = cache.get_stage_state(context["cache_db"], STAGE, CHECKPOINT)
    if saved is not None and saved.get("fingerprint") == context["fingerprint"]:
        context["logger"].info("[%s] возобновляю: отобрано %d, кандидатов %d",
                               STAGE, len(saved["selected"]), len(saved["candidates"]))
        return saved
    return {"fingerprint": context["fingerprint"], "selected": [seed],
            "candidates": [name for name in candidates if name != seed],
            "baseline": baseline, "trials": [], "added": 0,
            "rejections_in_row": 0, "since_rerank": 0, "round_sizes": []}


def _save_state(context: dict, state: dict) -> None:
    """Сохраняет состояние стадии после каждого решения."""
    cache.put_stage_state(context["cache_db"], STAGE, CHECKPOINT, state)


def _speculation_width(executor, context: dict, config, history: list) -> int:
    """Потолок опережения и стартовая ширина раунда.

    Опережать имеет смысл ровно настолько, сколько кандидатов обычно успевает
    получить отказ до очередного приёма. Принятый кандидат меняет набор, и всё
    посчитанное дальше не просто выбрасывается — оно ещё и отнимает полосу
    памяти у той единственной оценки, которая раунд и завершает.
    """
    workers = getattr(executor, "_max_workers", 1) or 1
    by_pool = max(1, workers // max(1, len(context["folds"])))
    ceiling = min(by_pool, config.speculation_width)
    if not history:
        return ceiling, 1
    recent = history[-5:]
    return ceiling, max(1, min(ceiling, int(statistics.median(recent))))


def _log_decision(logger, name: str, profile: dict, verdict: dict, size: int) -> None:
    """Пишет в лог одно решение о кандидате."""
    logger.info(
        "[%s] %-42s выигрыш %+.6f фолдов %d/3 путь %-8s %s -> набор %d",
        STAGE, name, profile["weighted_improvement"], profile["improved_folds"],
        verdict["path"], verdict["reason"], size)


def run(context: dict, ranking: list, executor, stop_check) -> dict:
    """Наращивает набор признаков по рангу до исчерпания кандидатов."""
    logger = context["logger"]
    config = context["config"].stage3
    cached = cache.get_stage_result(context["cache_db"], STAGE, context["fingerprint"])
    if cached is not None:
        logger.info("[%s] итог стадии уже посчитан: %d признаков",
                    STAGE, len(cached["selected"]))
        return cached

    candidates = list(ranking)
    if config.top_k_candidates > 0:
        candidates = ranking[:config.top_k_candidates]
    if not candidates:
        raise ValueError("стадии 3 нечего отбирать: пустой список кандидатов")
    seed_feature = candidates[0]
    baseline = _evaluate_baseline(context, [seed_feature], executor, stop_check)
    logger.info("[%s] затравка %s, взвешенная метрика %.6f",
                STAGE, seed_feature, baseline["weighted"])
    state = _load_state(context, candidates, baseline, seed_feature)

    while state["candidates"]:
        if state["added"] >= config.max_features_to_add:
            logger.info("[%s] достигнут предел добавлений %d",
                        STAGE, config.max_features_to_add)
            break
        if state["rejections_in_row"] >= config.patience_no_improvement:
            logger.info("[%s] терпение исчерпано", STAGE)
            break
        accepted_name = None
        consumed = []
        units = _candidate_units(context, state["selected"], state["candidates"])
        ceiling, start = _speculation_width(executor, context, config,
                                            state.setdefault("round_sizes", []))
        stream = pool.iterate_in_order(executor, context["cache_db"], STAGE, units,
                                       logger, stop_check, ceiling, start)
        for name, unit_jobs, results in stream:
            metrics = jobs.aggregate(results, unit_jobs, context["folds"])
            profile = jobs.improvement_profile(state["baseline"], metrics,
                                               context["folds"])
            verdict = decide(profile, config)
            consumed.append(name)
            state["trials"].append({"feature": name, **profile, **verdict})
            _log_decision(logger, name, profile, verdict,
                          len(state["selected"]) + int(verdict["accepted"]))
            emit_event("candidate", stage=STAGE, feature=name, **profile, **verdict)
            if verdict["accepted"]:
                state["selected"].append(name)
                state["baseline"] = metrics
                state["added"] += 1
                state["since_rerank"] += 1
                state["rejections_in_row"] = 0
                accepted_name = name
                break
            state["rejections_in_row"] += 1
        stream.close()
        state["round_sizes"].append(len(consumed))
        # Проход по рангу односторонний: отвергнутый кандидат выбывает и к нему
        # не возвращаются, поэтому счёт растёт линейно, а не квадратично.
        state["candidates"] = [name for name in state["candidates"]
                               if name not in set(consumed)]
        if accepted_name is None:
            _save_state(context, state)
            break
        if state["since_rerank"] >= config.rerank_after_n_added:
            state["candidates"] = _rerank(context, state["selected"],
                                          state["candidates"], executor, stop_check)
            state["since_rerank"] = 0
            logger.info("[%s] остаток кандидатов переупорядочен", STAGE)
        _save_state(context, state)

    payload = {"selected": state["selected"], "baseline": state["baseline"],
               "trials": state["trials"], "seed_feature": seed_feature}
    cache.put_stage_result(context["cache_db"], STAGE, context["fingerprint"], payload)
    logger.info("[%s] отобрано %d признаков, метрика %.6f",
                STAGE, len(payload["selected"]), payload["baseline"]["weighted"])
    emit_event("stage_done", stage=STAGE, selected=len(payload["selected"]))
    return payload

Writing nsel_iz_noutbuka/nsel/stages/stage3_ranked_forward.py


### Стадия 5 — парное спасение (идёт после стадии 3)

Ищет признаки, которые полезны только вместе.
* Кандидаты — отвергнутые по порогу (не по защите), с выигрышем не ниже 20% порога приёма,
  не больше `top_rejected` лучших.
* Пары по возможности берутся из разных семей признаков (`feature_family`).
* Пара принимается при выигрыше не меньше `0.000075` (полтора одиночных порога) и хотя бы
  двух улучшенных фолдах. Всего можно добавить не больше `max_additions` пар.

In [16]:
%%writefile nsel_iz_noutbuka/nsel/stages/stage5_pair_rescue.py
"""Стадия 5: парное спасение почти прошедших кандидатов."""
from __future__ import annotations

import itertools
import types

from .. import cache, jobs, pool
from ..data import feature_family
from ..logging_setup import emit_event
from .stage3_ranked_forward import guards_pass

STAGE = "stage5_pair_rescue"
GUARD_REASONS = ("просадка фолда", "просадка сегмента нулей",
                 "просадка сегмента покупавших")


def near_miss_candidates(trials: list, stage3_config, stage5_config,
                         selected=()) -> list:
    """Отвергнутые кандидаты, чей выигрыш был близок к порогу приёмки.

    Уже отобранные исключаются: после перезапусков один и тот же признак мог
    попасть в историю и отвергнутым, и принятым, а пара с ним дала бы
    повторяющееся имя столбца.
    """
    taken = set(selected)
    floor = stage5_config.near_miss_fraction * stage3_config.min_improvement
    near = [trial for trial in trials
            if not trial.get("accepted")
            and trial.get("feature") not in taken
            and trial.get("reason") not in GUARD_REASONS
            and trial.get("weighted_improvement", 0.0) >= floor]
    near.sort(key=lambda trial: -trial["weighted_improvement"])
    seen = []
    for trial in near:
        if trial["feature"] not in seen:
            seen.append(trial["feature"])
        if len(seen) >= stage5_config.top_rejected:
            break
    return seen


def build_pairs(candidates: list, prefer_cross_family: bool) -> list:
    """Пары кандидатов; при возможности из разных семей признаков."""
    pairs = list(itertools.combinations(candidates, 2))
    if not prefer_cross_family:
        return pairs
    cross = [pair for pair in pairs
             if feature_family(pair[0]) != feature_family(pair[1])]
    if cross:
        return cross
    return pairs


def _evaluate_pair(context: dict, selected: list, pair: tuple, executor,
                   stop_check) -> dict:
    """Метрика набора вместе с парой кандидатов."""
    config = context["config"]
    job_list = jobs.build_jobs(context, selected + list(pair),
                               seed=config.random_state,
                               iterations=config.probe.iterations,
                               early_stopping=config.probe.early_stopping_rounds)
    results = pool.run_jobs_cached(executor, context["cache_db"], STAGE, job_list,
                                   context["logger"], stop_check)
    return jobs.aggregate(results, job_list, context["folds"])


def _pair_accepted(profile: dict, config) -> bool:
    """Правило приёма пары."""
    allowed, _reason = guards_pass(profile, config)
    if not allowed:
        return False
    return (profile["weighted_improvement"] >= config.min_improvement
            and profile["improved_folds"] >= config.min_improved_folds)


def run(context: dict, selected: list, baseline: dict, trials: list, executor,
        stop_check) -> dict:
    """Пробует добавить пары почти прошедших кандидатов."""
    logger = context["logger"]
    config = context["config"].stage5
    if not config.enabled:
        logger.info("[%s] выключено по умолчанию", STAGE)
        return {"selected": list(selected), "baseline": baseline, "added": [],
                "enabled": False}
    cached = cache.get_stage_result(context["cache_db"], STAGE, context["fingerprint"])
    if cached is not None:
        logger.info("[%s] итог стадии уже посчитан", STAGE)
        return cached

    candidates = near_miss_candidates(trials, context["config"].stage3, config,
                                      selected)
    pairs = build_pairs(candidates, config.prefer_cross_family)[:config.max_pairs]
    logger.info("[%s] почти прошедших %d, пар к проверке %d",
                STAGE, len(candidates), len(pairs))

    current = list(selected)
    current_baseline = baseline
    added = []
    guard_config = types.SimpleNamespace(
        min_improvement=config.min_improvement,
        min_improved_folds=config.min_improved_folds,
        max_allowed_fold_degradation=config.max_allowed_fold_degradation,
        max_zero_segment_degradation=float("inf"),
        max_positive_segment_degradation=float("inf"))
    for pair in pairs:
        if len(added) >= config.max_additions:
            break
        # Пары строятся один раз, поэтому один признак может встретиться в двух
        # парах. Если он уже взят, пара перестаёт быть парой: её порог
        # 0.000075 задан для двух признаков сразу, а не для одного.
        if any(name in current for name in pair):
            logger.info("[%s] %s + %s пропущена, признак уже выбран", STAGE,
                        pair[0], pair[1])
            continue
        metrics = _evaluate_pair(context, current, pair, executor, stop_check)
        profile = jobs.improvement_profile(current_baseline, metrics,
                                           context["folds"])
        accepted = _pair_accepted(profile, guard_config)
        logger.info("[%s] %s + %s выигрыш %+.6f фолдов %d -> %s", STAGE, pair[0],
                    pair[1], profile["weighted_improvement"],
                    profile["improved_folds"], accepted)
        if accepted:
            current = current + list(pair)
            current_baseline = metrics
            added.append(list(pair))

    payload = {"selected": current, "baseline": current_baseline, "added": added,
               "enabled": True, "candidates": candidates}
    cache.put_stage_result(context["cache_db"], STAGE, context["fingerprint"], payload)
    emit_event("stage_done", stage=STAGE, added=len(added))
    return payload

Writing nsel_iz_noutbuka/nsel/stages/stage5_pair_rescue.py


### Стадия 4 — проверка набора на сидах

* Весь набор и одна затравка считаются на сидах 42, 52 и 62. В отчёт идут среднее,
  медиана, разброс и выигрыш набора над затравкой. Набор проверяется целиком, а не
  каждый признак по отдельности.
* Опционально (`confirm_doubtful`) признаки, прошедшие впритык к порогу, проверяются
  точечно: набор с признаком против набора без него на каждом сиде. Признак остаётся,
  если выигрыш положителен хотя бы на двух сидах из трёх и медиана больше нуля.

In [17]:
%%writefile nsel_iz_noutbuka/nsel/stages/stage4_multiseed.py
"""Стадия 4: проверка НАБОРА на нескольких сидах и разбор сомнительных."""
from __future__ import annotations

import statistics

from .. import cache, jobs, pool
from ..logging_setup import emit_event

STAGE = "stage4_multiseed"


def _metrics_for_seed(context: dict, features: list, seed: int, executor,
                      stop_check) -> dict:
    """Взвешенная метрика набора на одном сиде."""
    config = context["config"]
    job_list = jobs.build_jobs(context, features, seed=seed,
                               iterations=config.probe.iterations,
                               early_stopping=config.probe.early_stopping_rounds)
    results = pool.run_jobs_cached(executor, context["cache_db"], STAGE, job_list,
                                   context["logger"], stop_check)
    return jobs.aggregate(results, job_list, context["folds"])


def _stability(values: dict) -> dict:
    """Разброс взвешенной метрики по сидам."""
    numbers = [item["weighted"] for item in values.values()]
    spread = 0.0
    if len(numbers) > 1:
        spread = statistics.stdev(numbers)
    best = min(values, key=lambda seed: values[seed]["weighted"])
    worst = max(values, key=lambda seed: values[seed]["weighted"])
    return {"mean": statistics.fmean(numbers), "median": statistics.median(numbers),
            "stdev": spread, "spread": max(numbers) - min(numbers),
            "best_seed": best, "worst_seed": worst,
            "best": values[best]["weighted"], "worst": values[worst]["weighted"]}


def is_doubtful(trial: dict, config, forward_config) -> bool:
    """Признак сомнителен, если прошёл впритык к порогу приёма."""
    if trial.get("path") == "stable":
        return config.confirm_stable_tiny_gains
    threshold = config.doubtful_improvement_factor * forward_config.min_improvement
    return trial.get("weighted_improvement", 0.0) < threshold


def _confirm_feature(context: dict, features: list, feature: str, executor,
                     stop_check) -> dict:
    """Подтверждает пользу признака: знак и устойчивость по сидам.

    Метрики сегментов на дополнительных сидах только записываются: права
    отклонить признак у них нет, если временные свидетельства его держат.
    """
    config = context["config"].stage4
    without = [name for name in features if name != feature]
    improvements = []
    segments = []
    for seed in config.seeds:
        full = _metrics_for_seed(context, features, seed, executor, stop_check)
        reduced = _metrics_for_seed(context, without, seed, executor, stop_check)
        profile = jobs.improvement_profile(reduced, full, context["folds"])
        improvements.append(profile["weighted_improvement"])
        segments.append({"seed": seed, "zero": profile["worst_zero_degradation"],
                         "positive": profile["worst_positive_degradation"]})
    positive_seeds = sum(1 for value in improvements if value > 0)
    median_improvement = statistics.median(improvements)
    keep = (positive_seeds >= config.confirmation_min_positive_seeds
            and median_improvement > config.confirmation_min_median_improvement)
    return {"feature": feature, "median_improvement": median_improvement,
            "mean_improvement": statistics.fmean(improvements),
            "improvements": improvements, "positive_seeds": positive_seeds,
            "segments": segments, "segment_mode": config.segment_mode,
            "keep": keep}


def run(context: dict, features: list, baseline_features: list, trials: list,
        executor, stop_check) -> dict:
    """Проверяет устойчивость набора к случайности модели."""
    logger = context["logger"]
    config = context["config"].stage4
    cached = cache.get_stage_result(context["cache_db"], STAGE, context["fingerprint"])
    if cached is not None:
        logger.info("[%s] итог стадии уже посчитан", STAGE)
        return cached
    if not config.enabled:
        return {"features": list(features), "per_seed": {}, "stability": {},
                "confirmations": [], "dropped": []}

    per_seed = {}
    per_seed_baseline = {}
    for seed in config.seeds:
        per_seed[seed] = _metrics_for_seed(context, features, seed, executor,
                                           stop_check)
        per_seed_baseline[seed] = _metrics_for_seed(context, baseline_features, seed,
                                                    executor, stop_check)
        logger.info("[%s] сид %d: метрика набора %.6f, затравки %.6f",
                    STAGE, seed, per_seed[seed]["weighted"],
                    per_seed_baseline[seed]["weighted"])

    stability = _stability(per_seed)
    improvement = statistics.fmean(
        [per_seed_baseline[seed]["weighted"] - per_seed[seed]["weighted"]
         for seed in config.seeds])
    logger.info("[%s] среднее %.6f медиана %.6f разброс %.6f выигрыш набора %+.6f",
                STAGE, stability["mean"], stability["median"], stability["spread"],
                improvement)

    confirmations = []
    dropped = []
    kept = list(features)
    if config.confirm_doubtful:
        forward = context["config"].stage3
        accepted = set(features)
        doubtful = [trial["feature"] for trial in trials
                    if trial.get("accepted") and trial["feature"] in accepted
                    and is_doubtful(trial, config, forward)]
        logger.info("[%s] сомнительных признаков %d", STAGE, len(doubtful))
        for feature in doubtful:
            verdict = _confirm_feature(context, kept, feature, executor, stop_check)
            confirmations.append(verdict)
            logger.info("[%s] %-42s медиана %+.6f, сидов в плюс %d/%d -> %s",
                        STAGE, feature, verdict["median_improvement"],
                        verdict["positive_seeds"], len(config.seeds),
                        verdict["keep"])
            if not verdict["keep"]:
                kept = [name for name in kept if name != feature]
                dropped.append(feature)

    payload = {"features": kept, "dropped": dropped,
               "per_seed": {str(seed): per_seed[seed] for seed in per_seed},
               "stability": {**stability, "set_improvement": improvement},
               "confirmations": confirmations}
    cache.put_stage_result(context["cache_db"], STAGE, context["fingerprint"], payload)
    emit_event("stage_done", stage=STAGE, features=len(kept), dropped=len(dropped))
    return payload

Writing nsel_iz_noutbuka/nsel/stages/stage4_multiseed.py


### Стадия 6 — обратная чистка (ровно одна, в конце)

* База пересчитывается заново, потому что стадия 4 могла снять признаки.
* Признаки проверяются в порядке, обратном добавлению. Признак удаляется, если без него
  метрика не хуже чем на `neutral_tolerance` и не нарушены те же защиты, что при приёме.
* Здесь тоже используется счёт на опережение: пока решается судьба одного изъятия,
  следующие уже считаются.

In [18]:
%%writefile nsel_iz_noutbuka/nsel/stages/stage6_backward.py
"""Обратная чистка: один проход по набору в самом конце."""
from __future__ import annotations

import numpy as np

from .. import cache, jobs, pool
from ..logging_setup import emit_event
from .stage3_ranked_forward import guards_pass

STAGE = "stage6_backward"
CHECKPOINT = "progress"


def removal_order(features: list) -> list:
    """Порядок проверки — обратный порядку добавления, как у коллеги."""
    return list(reversed(features))


def removal_allowed(profile: dict, config, guards) -> tuple:
    """Решение об изъятии признака: нейтральность плюс те же защиты, что и приём."""
    if profile["weighted_improvement"] < -config.neutral_tolerance:
        return False, "изъятие ухудшает метрику"
    allowed, reason = guards_pass(profile, guards)
    if not allowed:
        return False, reason
    return True, "нейтрально"


def _removal_units(context: dict, features: list, order: list) -> list:
    """Единицы работы «набор без признака» в порядке проверки."""
    config = context["config"]
    units = []
    for name in order:
        reduced = [item for item in features if item != name]
        units.append((name, jobs.build_jobs(
            context, reduced, seed=config.random_state,
            iterations=config.probe.iterations,
            early_stopping=config.probe.early_stopping_rounds)))
    return units


def _baseline(context: dict, features: list, executor, stop_check) -> dict:
    """Метрика набора, пришедшего в чистку."""
    config = context["config"]
    job_list = jobs.build_jobs(context, features, seed=config.random_state,
                               iterations=config.probe.iterations,
                               early_stopping=config.probe.early_stopping_rounds)
    results = pool.run_jobs_cached(executor, context["cache_db"], STAGE, job_list,
                                   context["logger"], stop_check)
    return jobs.aggregate(results, job_list, context["folds"])


def run(context: dict, features: list, executor, stop_check) -> dict:
    """Удаляет признаки, чьё изъятие нейтрально и не нарушает защиту."""
    logger = context["logger"]
    config = context["config"].backward
    if not config.enabled:
        baseline = _baseline(context, features, executor, stop_check)
        return {"features": list(features), "baseline": baseline, "removed": []}
    cached = cache.get_stage_result(context["cache_db"], STAGE, context["fingerprint"])
    if cached is not None:
        logger.info("[%s] итог стадии уже посчитан", STAGE)
        return cached

    current = list(features)
    # База считается заново: стадия 4 могла снять сомнительные признаки, и
    # метрика, пришедшая из отбора, относилась бы уже к другому набору.
    current_baseline = _baseline(context, current, executor, stop_check)
    removed = []
    guards = context["config"].stage3
    pending = removal_order(current)

    while pending and len(current) > 1:
        units = _removal_units(context, current, pending)
        workers = getattr(executor, "_max_workers", 1) or 1
        width = max(config.speculation_width,
                    max(1, workers // max(1, len(context["folds"]))))
        stream = pool.iterate_in_order(executor, context["cache_db"], STAGE, units,
                                       logger, stop_check, width)
        removed_name = None
        checked = []
        for name, unit_jobs, results in stream:
            checked.append(name)
            metrics = jobs.aggregate(results, unit_jobs, context["folds"])
            profile = jobs.improvement_profile(current_baseline, metrics,
                                               context["folds"])
            drop, reason = removal_allowed(profile, config, guards)
            logger.info("[%s] без %-42s метрика %+.6f %s -> %s", STAGE, name,
                        -profile["weighted_improvement"], reason, drop)
            if drop:
                current = [item for item in current if item != name]
                current_baseline = metrics
                removed.append(name)
                removed_name = name
                break
        stream.close()
        pending = [name for name in pending if name not in set(checked)]
        cache.put_stage_state(context["cache_db"], STAGE, CHECKPOINT,
                              {"fingerprint": context["fingerprint"],
                               "features": current, "removed": removed})
        if removed_name is None:
            break

    payload = {"features": current, "baseline": current_baseline, "removed": removed}
    cache.put_stage_result(context["cache_db"], STAGE, context["fingerprint"], payload)
    logger.info("[%s] удалено %d, осталось %d", STAGE, len(removed), len(current))
    emit_event("stage_done", stage=STAGE, removed=len(removed), kept=len(current))
    return payload

Writing nsel_iz_noutbuka/nsel/stages/stage6_backward.py


### Стадия 7 — Optuna (по умолчанию выключена)

Подбор параметров щупа на той же кросс-валидации, что и отбор. Сначала разведка по
широкому пространству (TPE), потом пространство сужается по 20 лучшим точкам с запасом
и защитой границ, и идёт уточнение. Точки считаются пачками ask/tell: пачка целиком
уходит в пул без барьера внутри.

In [19]:
%%writefile nsel_iz_noutbuka/nsel/stages/stage7_optuna.py
"""Оптуна: разведка, затем сужение пространства и уточнение."""
from __future__ import annotations

import dataclasses

import numpy as np

from .. import cache, jobs, pool
from ..config import ProbeConfig
from ..logging_setup import emit_event

STAGE = "stage7_optuna"

SEARCH_SPACE = {
    "learning_rate": {"low": 0.01, "high": 0.30, "log": True, "kind": "float"},
    "depth": {"low": 4, "high": 10, "log": False, "kind": "int"},
    "l2_leaf_reg": {"low": 0.5, "high": 30.0, "log": True, "kind": "float"},
    "subsample": {"low": 0.5, "high": 1.0, "log": False, "kind": "float"},
    "colsample": {"low": 0.5, "high": 1.0, "log": False, "kind": "float"},
    "min_data_in_leaf": {"low": 1, "high": 200, "log": True, "kind": "int"},
}


def _sampler(config, startup_trials: int):
    """Создаёт TPE-сэмплер с настройками из промпта."""
    import optuna

    return optuna.samplers.TPESampler(
        n_startup_trials=startup_trials, multivariate=config.tpe_multivariate,
        group=config.tpe_group, seed=config.random_state)


def _suggest(trial, space: dict) -> dict:
    """Берёт у Оптуны одну точку пространства."""
    params = {}
    for name, bounds in space.items():
        if bounds["kind"] == "int":
            params[name] = trial.suggest_int(name, int(bounds["low"]),
                                             int(bounds["high"]), log=bounds["log"])
        else:
            params[name] = trial.suggest_float(name, bounds["low"], bounds["high"],
                                               log=bounds["log"])
    return params


def narrow_space(space: dict, records: list, config) -> dict:
    """Сужает пространство вокруг лучших точек разведки."""
    if len(records) < 2:
        return dict(space)
    ordered = sorted(records, key=lambda item: item["value"])[:config.adaptive_top_k]
    narrowed = {}
    for name, bounds in space.items():
        values = np.array([item["params"][name] for item in ordered], dtype=np.float64)
        low = float(np.quantile(values, config.adaptive_quantile_low))
        high = float(np.quantile(values, config.adaptive_quantile_high))
        span = bounds["high"] - bounds["low"]
        pad = config.adaptive_range_expand * max(high - low, 0.0)
        low, high = low - pad, high + pad
        minimum = config.adaptive_min_range_fraction * span
        if high - low < minimum:
            middle = (high + low) / 2.0
            low, high = middle - minimum / 2.0, middle + minimum / 2.0
        guard = config.adaptive_boundary_guard_fraction * span
        if low - bounds["low"] < guard:
            low = bounds["low"]
        if bounds["high"] - high < guard:
            high = bounds["high"]
        low = max(low, bounds["low"])
        high = min(high, bounds["high"])
        narrowed[name] = {**bounds, "low": low, "high": high}
        if bounds["kind"] == "int":
            narrowed[name]["low"] = int(np.floor(low))
            narrowed[name]["high"] = max(int(np.ceil(high)), int(np.floor(low)) + 1)
    return narrowed


def _probe_from_params(base: ProbeConfig, params: dict, config) -> ProbeConfig:
    """Собирает параметры щупа из точки пространства."""
    return dataclasses.replace(
        base, learning_rate=params["learning_rate"], depth=int(params["depth"]),
        l2_leaf_reg=params["l2_leaf_reg"], subsample=params["subsample"],
        colsample=params["colsample"],
        min_data_in_leaf=int(params["min_data_in_leaf"]),
        early_stopping_rounds=config.early_stopping_rounds)


def _evaluate_batch(context: dict, features: list, points: list, executor,
                    stop_check) -> list:
    """Считает пачку точек пространства целиком, без барьера внутри пачки."""
    config = context["config"]
    per_point = []
    all_jobs = []
    for params in points:
        probe = _probe_from_params(config.probe, params, config.optuna)
        trial_context = {**context, "config": dataclasses.replace(config, probe=probe)}
        job_list = jobs.build_jobs(trial_context, features,
                                   seed=config.optuna.random_state,
                                   iterations=probe.iterations,
                                   early_stopping=probe.early_stopping_rounds)
        per_point.append(job_list)
        all_jobs.extend(job_list)
    results = pool.run_jobs_cached(executor, context["cache_db"], STAGE, all_jobs,
                                   context["logger"], stop_check)
    return [jobs.aggregate(results, job_list, context["folds"])["weighted"]
            for job_list in per_point]


def _run_phase(context: dict, features: list, space: dict, trials: int,
               startup: int, executor, stop_check, phase: str) -> list:
    """Проводит одну фазу поиска пачками ask/tell."""
    import optuna

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    logger = context["logger"]
    config = context["config"].optuna
    study = optuna.create_study(direction="minimize",
                                sampler=_sampler(config, startup))
    batch = max(1, getattr(executor, "_max_workers", 1) // max(1, len(context["folds"])))
    records = []
    while len(records) < trials:
        size = min(batch, trials - len(records))
        asked = [study.ask() for _ in range(size)]
        points = [_suggest(trial, space) for trial in asked]
        values = _evaluate_batch(context, features, points, executor, stop_check)
        for trial, params, value in zip(asked, points, values):
            study.tell(trial, value)
            records.append({"params": params, "value": value, "phase": phase})
        best = min(record["value"] for record in records)
        logger.info("[%s] %s %d/%d лучший %.6f", STAGE, phase, len(records),
                    trials, best)
    return records


def run(context: dict, features: list, executor, stop_check) -> dict:
    """Ищет параметры модели на той же кросс-валидации, что и отбор."""
    logger = context["logger"]
    config = context["config"].optuna
    if not config.enabled:
        logger.info("[%s] выключено", STAGE)
        return {"enabled": False}
    cached = cache.get_stage_result(context["cache_db"], STAGE, context["fingerprint"])
    if cached is not None:
        logger.info("[%s] итог стадии уже посчитан", STAGE)
        return cached

    exploration = _run_phase(context, features, SEARCH_SPACE,
                             config.exploration_trials,
                             config.tpe_startup_trials_explore, executor, stop_check,
                             "разведка")
    focused = []
    space = dict(SEARCH_SPACE)
    if config.adaptive_search:
        space = narrow_space(SEARCH_SPACE, exploration, config)
        logger.info("[%s] пространство сужено: %s", STAGE,
                    {name: (round(bounds["low"], 4), round(bounds["high"], 4))
                     for name, bounds in space.items()})
        focused = _run_phase(context, features, space, config.focused_trials,
                             config.tpe_startup_trials_focus, executor, stop_check,
                             "уточнение")

    records = exploration + focused
    best = min(records, key=lambda item: item["value"])
    payload = {"enabled": True, "best_params": best["params"],
               "best_value": best["value"], "trials": len(records),
               "narrowed_space": space,
               "exploration_best": min(item["value"] for item in exploration)}
    cache.put_stage_result(context["cache_db"], STAGE, context["fingerprint"], payload)
    logger.info("[%s] лучший счёт %.6f на %d трайлах", STAGE, best["value"],
                len(records))
    emit_event("stage_done", stage=STAGE, best=best["value"], trials=len(records))
    return payload

Writing nsel_iz_noutbuka/nsel/stages/stage7_optuna.py


### `nsel/orchestrator.py` — порядок стадий, остановка, итоговый отчёт

* `run_selection(config)` — главная функция: подготовка → стадия 1 на широком пуле →
  стадии 3, 5, 4, 6, 7 на узком пуле → проверки → `selected_features_<сочетание>.json`.
* Остановка: `Ctrl+C`, `SIGTERM` или файл `STOP` в каталоге прогона. Остановка
  срабатывает на границе задания, и посчитанное уже лежит в кэше.
* `run_stage_with_retry`: если процесс-исполнитель погиб, пул поднимается заново и стадия
  повторяется. Это дёшево, потому что всё посчитанное берётся из кэша.

In [20]:
%%writefile nsel_iz_noutbuka/nsel/orchestrator.py
"""Порядок стадий, возобновление и итоговый отчёт."""
from __future__ import annotations

import dataclasses
import json
import signal
import time
from pathlib import Path

import psutil

from . import cache, checks, data, pool
from .config import assert_fold_contract, build_folds
from .logging_setup import emit_event, setup_logging
from .stages import (
    stage1_null_importance,
    stage3_ranked_forward,
    stage4_multiseed,
    stage5_pair_rescue,
    stage6_backward,
    stage7_optuna,
)

_STOP = {"requested": False}


def request_stop(*_arguments) -> None:
    """Помечает прогон к остановке на ближайшей границе задания."""
    _STOP["requested"] = True


def install_signal_handlers() -> None:
    """Ставит обработчики, чтобы прерывание не теряло посчитанное."""
    signal.signal(signal.SIGINT, request_stop)
    signal.signal(signal.SIGTERM, request_stop)


def make_stop_check(run_dir):
    """Проверка остановки: сигнал или файл STOP в каталоге прогона."""
    stop_file = Path(run_dir) / "STOP"

    def check() -> bool:
        if _STOP["requested"]:
            return True
        return stop_file.exists()

    return check


def build_context(config, logger) -> dict:
    """Общее состояние прогона, передаваемое стадиям."""
    folds = build_folds()
    assert_fold_contract(folds)
    return {"config": config, "folds": folds, "logger": logger,
            "cache_db": config.cache_db, "fingerprint": config.fingerprint()}


def open_pool(config, logger, wide: bool):
    """Поднимает пул под конкретный класс стадий."""
    available = psutil.virtual_memory().available / 1024 ** 3
    workers = config.parallel.resolve_workers(psutil.cpu_count() or 4, available, wide)
    threads = config.parallel.threads_for(wide)
    kind = "узкий"
    if wide:
        kind = "широкий"
    logger.info("пул %s: процессов %d по %d потоков, свободно памяти %.1f ГБ",
                kind, workers, threads, available)
    return pool.make_pool(workers, threads)


def _write_report(config, report: dict) -> Path:
    """Пишет итог прогона в JSON рядом с логом."""
    path = Path(config.run_dir) / f"selected_features_{config.combination}.json"
    path.write_text(json.dumps(report, ensure_ascii=False, indent=2, default=str),
                    encoding="utf-8")
    return path


def run_stage_with_retry(config, logger, wide: bool, executor, action):
    """Выполняет стадию, переподнимая пул, если процесс-исполнитель погиб.

    Гибель воркера обесценивает пул целиком, но не посчитанное: каждая оценка
    уже лежит в кэше. Поэтому вторая попытка стоит дёшево и спасает прогон,
    который иначе оборвался бы на ровном месте.
    """
    try:
        return action(executor), executor
    except pool.WorkerLost as error:
        logger.warning("пул погиб (%s), поднимаю заново и повторяю стадию", error)
        executor.shutdown(wait=False, cancel_futures=True)
        fresh = open_pool(config, logger, wide)
        return action(fresh), fresh


def _stage_banner(logger, name: str, order: str) -> float:
    """Отмечает начало стадии в логе и возвращает момент старта."""
    logger.info("=" * 70)
    logger.info("СТАДИЯ %s: %s", order, name)
    return time.perf_counter()


def _stage_done(logger, name: str, started: float) -> None:
    """Отмечает конец стадии в логе."""
    logger.info("стадия %s заняла %.1fс", name, time.perf_counter() - started)


def run_selection(config) -> dict:
    """Проводит отбор целиком в порядке, заданном новыми требованиями."""
    Path(config.run_dir).mkdir(parents=True, exist_ok=True)
    logger = setup_logging(config.log_file, config.events_file)
    logger.info("сочетание %s, отпечаток %s", config.combination,
                config.fingerprint())
    cache.init_db(config.cache_db)
    install_signal_handlers()
    stop_check = make_stop_check(config.run_dir)
    context = build_context(config, logger)

    manifest = data.prepare_snapshots(
        config.snapshot_dir, config.prepared_dir,
        data.required_cutoffs(context["folds"]), users_limit=config.users_limit,
        feature_limit=config.feature_limit, logger=logger,
        drop_count_bins=config.drop_count_bin_categories,
        anchor_dir=config.anchor_dir or None)
    features = manifest["features"]

    stages_run = []
    # Стадия 1 учит на всех признаках сразу и держит одиннадцать гигабайт на
    # процесс; остальные стадии работают с наборами по десятку признаков и
    # укладываются в гигабайт. Один пул на весь прогон заставил бы их всех
    # жить по мерке самой прожорливой стадии.
    executor = open_pool(config, logger, wide=True)
    try:
        started = _stage_banner(logger, "временная null importance", "1")
        first, executor = run_stage_with_retry(
            config, logger, True, executor,
            lambda pool_: stage1_null_importance.run(context, features, pool_,
                                                     stop_check))
        stages_run.append(stage1_null_importance.STAGE)
        _stage_done(logger, "1", started)
    finally:
        executor.shutdown(wait=False, cancel_futures=True)

    executor = open_pool(config, logger, wide=False)
    try:
        started = _stage_banner(logger, "ranked forward selection", "3")
        forward, executor = run_stage_with_retry(
            config, logger, False, executor,
            lambda pool_: stage3_ranked_forward.run(context, first["passed"],
                                                    pool_, stop_check))
        stages_run.append(stage3_ranked_forward.STAGE)
        _stage_done(logger, "3", started)

        started = _stage_banner(logger, "парное спасение", "5")
        rescue, executor = run_stage_with_retry(
            config, logger, False, executor,
            lambda pool_: stage5_pair_rescue.run(context, forward["selected"],
                                                 forward["baseline"],
                                                 forward["trials"], pool_,
                                                 stop_check))
        stages_run.append(stage5_pair_rescue.STAGE)
        _stage_done(logger, "5", started)

        started = _stage_banner(logger, "проверка набора на сидах", "4")
        multiseed, executor = run_stage_with_retry(
            config, logger, False, executor,
            lambda pool_: stage4_multiseed.run(context, rescue["selected"],
                                               [forward["seed_feature"]],
                                               forward["trials"], pool_,
                                               stop_check))
        stages_run.append(stage4_multiseed.STAGE)
        _stage_done(logger, "4", started)

        started = _stage_banner(logger, "обратная чистка", "6")
        pruned, executor = run_stage_with_retry(
            config, logger, False, executor,
            lambda pool_: stage6_backward.run(context, multiseed["features"],
                                              pool_, stop_check))
        stages_run.append(stage6_backward.STAGE)
        _stage_done(logger, "6", started)

        optuna_result = stage7_optuna.run(context, pruned["features"], executor,
                                          stop_check)
        if optuna_result.get("enabled"):
            stages_run.append(stage7_optuna.STAGE)
    finally:
        executor.shutdown(wait=False, cancel_futures=True)

    report = {"combination": config.combination,
              "fingerprint": config.fingerprint(),
              "folds": [fold.as_dict() for fold in context["folds"]],
              "probe_learning_rate": config.probe.learning_rate,
              "feature_pool": len(features),
              "stage1_passed": len(first["passed"]),
              "stage1_reason": first["reason"],
              "forward_selected": forward["selected"],
              "seed_feature": forward["seed_feature"],
              "pair_rescue": rescue,
              "multiseed": multiseed,
              "backward_removed": pruned["removed"],
              "selected_features": pruned["features"],
              "final_metric": pruned["baseline"]["weighted"],
              "per_fold": pruned["baseline"]["per_fold"],
              "optuna": optuna_result,
              "config": config.decision_payload()}
    report["checks"] = checks.run_all(report, config, context["folds"], stages_run)
    path = _write_report(config, report)
    logger.info("итог: признаков %d, метрика %.6f, файл %s",
                len(report["selected_features"]), report["final_metric"], path)
    emit_event("run_done", features=len(report["selected_features"]),
               metric=report["final_metric"])
    return report

Writing nsel_iz_noutbuka/nsel/orchestrator.py


### `run_nsel.py` — запуск из командной строки

Тот же `run_selection`, только параметры приходят из аргументов. Примеры есть в разделе 5.

In [21]:
%%writefile nsel_iz_noutbuka/run_nsel.py
"""Запуск отбора: боевой прогон или быстрый прогон на подвыборке."""
from __future__ import annotations

import argparse
import sys
import dataclasses
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

from nsel import config as config_module
from nsel.orchestrator import run_selection

DEFAULT_SNAPSHOTS = "/Volumes/ECupOzon/backups/server_cpu_final_20260828/ozon_snapshot_cache_v8"


def parse_arguments():
    """Разбирает аргументы командной строки."""
    parser = argparse.ArgumentParser(description="отбор признаков, схема трёх фолдов")
    parser.add_argument("--run-dir", required=True)
    parser.add_argument("--snapshot-dir", default=DEFAULT_SNAPSHOTS)
    parser.add_argument("--model", default="catboost", choices=config_module.MODEL_MODES)
    parser.add_argument("--target", default="log_level",
                        choices=config_module.TARGET_MODES)
    parser.add_argument("--anchor-dir", default="",
                        help="каталог с якорем segment_adaptive")
    parser.add_argument("--smoke", action="store_true")
    parser.add_argument("--users", type=int, default=0)
    parser.add_argument("--features", type=int, default=0)
    parser.add_argument("--workers", type=int, default=0)
    parser.add_argument("--threads", type=int, default=2)
    parser.add_argument("--pair-rescue", action="store_true")
    parser.add_argument("--confirm-doubtful", action="store_true")
    parser.add_argument("--optuna", action="store_true")
    parser.add_argument("--null-runs", type=int, default=0)
    parser.add_argument("--actual-runs", type=int, default=0)
    parser.add_argument("--top-k", type=int, default=0)
    parser.add_argument("--max-features", type=int, default=0)
    parser.add_argument("--probe-iterations", type=int, default=0)
    parser.add_argument("--probe-lr", type=float, default=0.0)
    parser.add_argument("--speculation", type=int, default=0)
    parser.add_argument("--memory-per-worker", type=float, default=0.0)
    parser.add_argument("--prepared-dir", default="")
    return parser.parse_args()


def apply_scale_overrides(overrides: dict, base, arguments) -> dict:
    """Точечно урезает объём счёта, не трогая правила отбора."""
    stage1 = dict(actual_runs=arguments.actual_runs or base.stage1.actual_runs,
                  null_runs=arguments.null_runs or base.stage1.null_runs)
    overrides["stage1"] = dataclasses.replace(base.stage1, **stage1)
    stage3 = dict(top_k_candidates=arguments.top_k or base.stage3.top_k_candidates,
                  max_features_to_add=(arguments.max_features
                                       or base.stage3.max_features_to_add),
                  speculation_width=(arguments.speculation
                                     or base.stage3.speculation_width))
    overrides["stage3"] = dataclasses.replace(base.stage3, **stage3)
    probe = dict(iterations=arguments.probe_iterations or base.probe.iterations,
                 learning_rate=arguments.probe_lr or base.probe.learning_rate)
    overrides["probe"] = dataclasses.replace(base.probe, **probe)
    return overrides


def build_config(arguments):
    """Собирает конфиг прогона по аргументам."""
    parallel = {"n_workers": arguments.workers,
                "threads_per_worker": arguments.threads}
    if arguments.memory_per_worker:
        parallel["memory_per_worker_gb"] = arguments.memory_per_worker
    overrides = {"model_mode": arguments.model, "target_mode": arguments.target,
                 "parallel": config_module.ParallelConfig(**parallel)}
    if arguments.pair_rescue:
        overrides["stage5"] = config_module.Stage5Config(enabled=True)
    if arguments.confirm_doubtful:
        overrides["stage4"] = config_module.Stage4Config(confirm_doubtful=True)
    if arguments.optuna:
        overrides["optuna"] = config_module.OptunaConfig(enabled=True)
    if arguments.smoke:
        base = config_module.smoke_config(arguments.run_dir, arguments.snapshot_dir)
    else:
        base = config_module.RunConfig(run_dir=Path(arguments.run_dir),
                                       snapshot_dir=Path(arguments.snapshot_dir))
    if arguments.anchor_dir:
        overrides["anchor_dir"] = arguments.anchor_dir
    if arguments.prepared_dir:
        overrides["shared_prepared_dir"] = arguments.prepared_dir
    if arguments.users:
        overrides["users_limit"] = arguments.users
    if arguments.features:
        overrides["feature_limit"] = arguments.features
    overrides = apply_scale_overrides(overrides, base, arguments)
    return dataclasses.replace(base, **overrides)


def main() -> int:
    """Точка входа."""
    arguments = parse_arguments()
    report = run_selection(build_config(arguments))
    print(f"признаков {len(report['selected_features'])}, "
          f"метрика {report['final_metric']:.6f}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing nsel_iz_noutbuka/run_nsel.py


### `tools/build_adaptive_anchor.py` — якоря адаптивных таргетов (приложение)

Нужен только для таргетов `segment_adaptive`, `volatility_adaptive`, `personal_hard` и
`segment_logspace`. Логика якоря не переписана, а берётся из конвейера Владимира:
скрипт подгружает его файл через `--module`. В ноутбуке этот скрипт только записывается
на диск и не запускается.

In [22]:
%%writefile nsel_iz_noutbuka/tools/build_adaptive_anchor.py
"""Расчёт якоря segment_adaptive для каждого среза.

Якоря нет в готовых срезах, он строится по сырой истории: двенадцать блоков по
тридцать дней, псевдобэктест по каждому масштабу и выбор для каждого сегмента
пользователей того масштаба, что исторически ошибался меньше.

Логика не переписывается, а берётся из модуля коллеги: любое переписывание
такого объёма это способ незаметно разойтись с его числами.
"""
from __future__ import annotations

import argparse
import importlib.util
import sys
import time
from pathlib import Path

# Функция коллеги считает все адаптивные якоря разом, поэтому сохраняем их
# все: пересчёт ради одного столбца стоил бы столько же.
ANCHOR_COLUMNS = ("anchor_segment_adaptive", "anchor_volatility_adaptive",
                  "anchor_segment_logspace_log", "anchor_segment_logspace",
                  "anchor_personal_hard", "anchor_frequency_monetary_adaptive")
SEGMENT_COLUMNS = ("gmv_30d", "gmv_60d", "purchase_days_90d",
                   "days_since_last_purchase")
INDEX_COLUMN = "user_id"
# to_ord нужен соседнему якорю: обёртка коллеги считает все адаптивные
# якоря разом, и без этого столбца падает целиком.
RAW_COLUMNS = ("user_id", "event_date", "gmv", "to_ord")


def load_colleague_module(path):
    """Загружает модуль коллеги как обычный модуль по пути к файлу."""
    spec = importlib.util.spec_from_file_location("ozon_colleague", str(path))
    module = importlib.util.module_from_spec(spec)
    # Регистрация до выполнения обязательна: иначе dataclass внутри модуля не
    # находит собственное пространство имён и падает.
    sys.modules["ozon_colleague"] = module
    spec.loader.exec_module(module)
    return module


def cutoff_from_name(path) -> str:
    """Дата среза из имени файла snapshot_YYYYMMDD.parquet."""
    stem = Path(path).stem.split("_")[-1]
    return f"{stem[:4]}-{stem[4:6]}-{stem[6:8]}"


def read_snapshot(path, users_limit: int):
    """Срез с полями, нужными для сегментов, индексированный пользователем."""
    import pandas as pd

    frame = pd.read_parquet(path, columns=[INDEX_COLUMN, *SEGMENT_COLUMNS])
    # В срезах пользователь записан категорией из строк, в сырой истории целым
    # числом. Без приведения соединение не находит никого и якорь выходит
    # нулевым у всех, причём молча.
    frame[INDEX_COLUMN] = frame[INDEX_COLUMN].astype(str).astype("int64")
    frame = frame.sort_values(INDEX_COLUMN).set_index(INDEX_COLUMN)
    if users_limit:
        frame = frame.iloc[:users_limit]
    return frame


def build_one(module, raw, snapshot, cutoff):
    """Все адаптивные якоря одного среза и краткая сводка о них."""
    anchors, diagnostics = module._build_adaptive_target_anchors(
        raw, snapshot, cutoff)
    missing = [name for name in ANCHOR_COLUMNS if name not in anchors.columns]
    if missing:
        raise KeyError(f"в результате нет якорей {missing}")
    return anchors[list(ANCHOR_COLUMNS)], diagnostics


def main(argv=None) -> int:
    """Считает якорь для всех срезов каталога и кладёт рядом отдельным файлом."""
    import pandas as pd

    parser = argparse.ArgumentParser(description="якорь segment_adaptive")
    parser.add_argument("--raw", required=True, help="сырая история с gmv")
    parser.add_argument("--snapshot-dir", required=True)
    parser.add_argument("--out-dir", required=True)
    parser.add_argument("--module", required=True, help="файл конвейера коллеги")
    parser.add_argument("--users", type=int, default=0)
    parser.add_argument("--cutoffs", default="", help="через запятую, пусто это все")
    arguments = parser.parse_args(argv)

    module = load_colleague_module(arguments.module)
    out_dir = Path(arguments.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    started = time.time()
    raw = pd.read_parquet(arguments.raw, columns=list(RAW_COLUMNS))
    print(f"[сырьё] строк {len(raw)}, прочитано за {time.time() - started:.0f} с")

    wanted = {c for c in arguments.cutoffs.split(",") if c}
    files = sorted(Path(arguments.snapshot_dir).glob("snapshot_*.parquet"))
    written = 0
    for path in files:
        cutoff = cutoff_from_name(path)
        if wanted and cutoff not in wanted:
            continue
        moment = time.time()
        snapshot = read_snapshot(path, arguments.users)
        anchors, _ = build_one(module, raw, snapshot, cutoff)
        target = out_dir / f"anchor_segment_adaptive_{cutoff.replace('-', '')}.parquet"
        anchors.reset_index().to_parquet(target, index=False)
        parts = []
        for name in ANCHOR_COLUMNS:
            values = anchors[name]
            # Логарифмический якорь законно бывает нулевым у неактивных, а вот
            # нулевой у всех означает, что идентификаторы не совпали.
            if int((values <= 0).sum()) == len(values):
                raise ValueError(
                    f"[{cutoff}] якорь {name} нулевой у всех: почти наверняка "
                    "не совпали идентификаторы среза и сырой истории")
            parts.append(f"{name.replace('anchor_', '')} {values.mean():.1f}")
        print(f"[{cutoff}] пользователей {len(anchors)}, "
              f"{', '.join(parts)}, за {time.time() - moment:.0f} с")
        written += 1
    print(f"готово: файлов {written}, всего {time.time() - started:.0f} с")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing nsel_iz_noutbuka/tools/build_adaptive_anchor.py


### Импорт пакета

Модули выше записаны на диск. Импортируем их так же, как обычный пакет. Повторный запуск
ячейки подхватывает изменения в коде.

In [23]:
import importlib
import sys

sys.path.insert(0, str(Path("nsel_iz_noutbuka").resolve()))
for name in [name for name in sys.modules if name == "nsel" or name.startswith("nsel.")]:
    del sys.modules[name]
importlib.invalidate_caches()

from nsel import config as C
from nsel.orchestrator import run_selection
print("nsel импортирован из", Path(C.__file__).parent)

nsel импортирован из /Volumes/ECupOzon/tmp/nsel_otbor_priznakov_20260830/nsel_iz_noutbuka/nsel


## 3. Результаты боевых прогонов

Отбор шёл на полном объёме: 250 тыс. пользователей, 909 признаков, три фолда (ноябрь 0.25, декабрь 0.35, январь 0.40). Метрика — взвешенный RMSLE щупа на итоговом наборе, сортировка по ней. «Разброс сидов» — разница лучшего и худшего из сидов 42/52/62 на итоговом наборе.

| сочетание | признаков | метрика | ноябрь | декабрь | январь | в пуле | после стадии 1 | разброс сидов | пар | снято чисткой |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| xgboost_log_level | 45 | 1.714366 | 1.73546 | 1.73926 | 1.67940 | 909 | 273 | 0.000092 | 2 | 5 |
| catboost_log_level | 33 | 1.714631 | 1.73579 | 1.73952 | 1.67962 | 909 | 284 | 0.000273 | 2 | 4 |
| catboost_log_diff | 54 | 1.715168 | 1.73511 | 1.74205 | 1.67918 | 909 | 348 | 0.001678 | 1 | 0 |
| lightgbm_log_level | 49 | 1.715356 | 1.73591 | 1.74021 | 1.68077 | 909 | 319 | 0.000250 | 2 | 1 |
| xgboost_log_diff | 40 | 1.716881 | 1.73705 | 1.74207 | 1.68223 | 909 | 270 | 0.000048 | 2 | 10 |
| lightgbm_log_diff | 45 | 1.718170 | 1.73705 | 1.74335 | 1.68434 | 909 | 317 | 0.000283 | 1 | 5 |
| catboost_segment_adaptive | 48 | 1.720539 | 1.74055 | 1.74332 | 1.68810 | 909 | 450 | 0.000710 | 2 | 4 |
| catboost_segment_logspace | 49 | 1.721457 | 1.74226 | 1.74720 | 1.68593 | 909 | 311 | 0.000807 | 0 | 3 |
| xgboost_segment_adaptive | 41 | 1.722787 | 1.74169 | 1.74774 | 1.68914 | 909 | 275 | 0.000225 | 2 | 4 |
| catboost_volatility_adaptive | 49 | 1.731682 | 1.74218 | 1.75446 | 1.70519 | 909 | 366 | 0.000342 | 0 | 1 |
| xgboost_volatility_adaptive | 45 | 1.735041 | 1.74609 | 1.75622 | 1.70961 | 909 | 283 | 0.000782 | 0 | 0 |
| xgboost_personal_hard | 54 | 1.779188 | 1.79786 | 1.80204 | 1.74752 | 909 | 276 | 0.000594 | 0 | 0 |

In [24]:
# Та же таблица из файлов results/<сочетание>/selected_features_*.json рядом с ноутбуком
import json
import pandas as pd

rows = []
for path in sorted(Path("results").glob("*/selected_features_*.json")):
    report = json.loads(path.read_text(encoding="utf-8"))
    folds = report["per_fold"]
    rows.append({
        "сочетание": report["combination"],
        "признаков": len(report["selected_features"]),
        "метрика": round(report["final_metric"], 6),
        "ноябрь": round(folds["nov_2025"]["rmsle"], 5),
        "декабрь": round(folds["dec_2025"]["rmsle"], 5),
        "январь": round(folds["jan_2026"]["rmsle"], 5),
        "после стадии 1": report["stage1_passed"],
        "разброс сидов": round(report["multiseed"]["stability"]["spread"], 6),
        "пар добавлено": len(report["pair_rescue"].get("added", [])),
        "снято чисткой": len(report["backward_removed"]),
    })
table = pd.DataFrame(rows).sort_values("метрика").reset_index(drop=True) if rows else pd.DataFrame()
table if rows else print("папки results/ рядом нет")

,сочетание,признаков,метрика,ноябрь,декабрь,январь,после стадии 1,разброс сидов,пар добавлено,снято чисткой
0,xgboost_log_level,45,1.714366,1.73546,1.73926,1.67940,273,0.000092,2,5
1,catboost_log_level,33,1.714631,1.73579,1.73952,1.67962,284,0.000273,2,4
2,catboost_log_diff,54,1.715168,1.73511,1.74205,1.67918,348,0.001678,1,0
3,lightgbm_log_level,49,1.715356,1.73591,1.74021,1.68077,319,0.000250,2,1
4,xgboost_log_diff,40,1.716881,1.73705,1.74207,1.68223,270,0.000048,2,10
5,lightgbm_log_diff,45,1.718170,1.73705,1.74335,1.68434,317,0.000283,1,5
6,catboost_segment_adaptive,48,1.720539,1.74055,1.74332,1.68810,450,0.000710,2,4
7,catboost_segment_logspace,49,1.721457,1.74226,1.74720,1.68593,311,0.000807,0,3
8,xgboost_segment_adaptive,41,1.722787,1.74169,1.74774,1.68914,275,0.000225,2,4
9,catboost_volatility_adaptive,49,1.731682,1.74218,1.75446,1.70519,366,0.000342,0,1


In [25]:
# Итоговый список признаков любого сочетания
COMBINATION = "catboost_log_level"
path = Path("results") / COMBINATION / f"selected_features_{COMBINATION}.json"
if path.exists():
    print("\n".join(json.loads(path.read_text(encoding="utf-8"))["selected_features"]))

cutoff_dayofyear_cos
gmv_search_sum_365d
any_cart_no_order_day_share_active_90d
search_only_share_active_90d
cart_per_search_180d
order_per_search_180d
purchase_days_block_90_179
gmv_per_purchase_day_7d
gmv_decay30_hl90
search_to_cart_block_rate_0_6
cyc_cur_search
square_log1p_decay_to_ord_half_life_60d
square_log1p_gmv_sum_14d
log1p_order_gap_mean_days
cyc_completed
square_log1p_decay_to_ord_half_life_14d
purchase_days_calendar_rate_365d
days_since_last_order
pct_rank_gmv_decay30_hl60
search_days_7d__denis
cyc_cart_excess
row_days_7d
cutoff_is_weekend
searches_share30_of_180
searches_sum_14d
searches_calendar_day_rate_14d
ph_gmv_cv
bayesian_search_to_order_conversion_365d
to_ord_share30_of_365
prior_freq90_money180
percentile_rank_gmv_sum_365d
percentile_rank_lifetime_gmv
gmv_per_order_30d__denis


## 4. Проверка на синтетике

100 пользователей, пять дат отсечения в том же формате, что и настоящие срезы:

* сигнал: `orders_cnt_30d`, `habit_score_90d`, `gmv_30d` (он же якорь) и категориальный
  `segment_kind`;
* шум: 20 числовых `noise_XX_30d`, категориальный `device_type`, календарный `cutoff_month_cat`;
* `orders_cnt_30d_cat` — биннинговая категория, отбор должен её отбросить ещё на подготовке;
* таргет с нулями: покупает ли пользователь, зависит от склонности и сегмента, а сумма
  логнормальная.

Правила отбора те же, урезан только объём счёта (`smoke_config`: меньше null-прогонов и
деревьев), и процессов 4.

In [26]:
import numpy as np
import polars as pl

CUTOFFS = ("2025-09-16", "2025-10-16", "2025-11-15", "2025-12-15", "2026-01-14")
ADAPTIVE = ("anchor_segment_adaptive", "anchor_volatility_adaptive",
            "anchor_personal_hard", "anchor_segment_logspace_log")
DEMO = Path("demo_sintetika")


def make_synthetic(root, users: int = 100, noise: int = 20, seed: int = 0) -> None:
    """Пишет snapshot_*.parquet и файлы якорей адаптивных таргетов."""
    root = Path(root)
    snapshots, anchors = root / "snapshots", root / "anchors"
    snapshots.mkdir(parents=True, exist_ok=True)
    anchors.mkdir(parents=True, exist_ok=True)
    generator = np.random.default_rng(seed)
    ids = np.arange(1, users + 1)
    habit = generator.normal(0, 1, users)
    segment = generator.choice(["a", "b", "c"], users)
    for cutoff in CUTOFFS:
        drift = habit + generator.normal(0, 0.3, users)
        orders = generator.poisson(np.exp(0.5 * drift)).astype(np.int64)
        gmv_30d = np.where(orders > 0, orders * np.exp(6 + 0.4 * drift), 0.0)
        frame = {
            "__snapshot_index_user_id__": ids,
            "orders_cnt_30d": orders,
            "habit_score_90d": drift + generator.normal(0, 0.2, users),
            "gmv_30d": gmv_30d,
            "gmv_decay30_hl30": gmv_30d * 0.8,
            "segment_kind": segment,
            "device_type": generator.choice(["ios", "android", "web"], users),
            "cutoff_month_cat": np.full(users, cutoff[5:7]),
            "orders_cnt_30d_cat": np.where(orders > 1, "many", "few"),
        }
        for index in range(noise):
            frame[f"noise_{index:02d}_30d"] = generator.normal(0, 1, users)
        buy = generator.random(users) < 1 / (1 + np.exp(-(3 * drift + 1.5 * (segment == "a"))))
        amount = np.exp(6.5 + 0.6 * drift + generator.normal(0, 0.5, users))
        frame["target_30d"] = np.where(buy, amount, 0.0)
        stamp = cutoff.replace("-", "")
        pl.DataFrame(frame).write_parquet(snapshots / f"snapshot_{stamp}.parquet")
        anchor = {"user_id": ids}
        for name in ADAPTIVE:
            anchor[name] = np.log1p(gmv_30d) if name.endswith("_log") else gmv_30d * 0.9
        pl.DataFrame(anchor).write_parquet(anchors / f"anchor_segment_adaptive_{stamp}.parquet")


import shutil
shutil.rmtree(DEMO, ignore_errors=True)
make_synthetic(DEMO)
pl.read_parquet(DEMO / "snapshots" / "snapshot_20260114.parquet").head()

__snapshot_index_user_id__,orders_cnt_30d,habit_score_90d,gmv_30d,gmv_decay30_hl30,segment_kind,device_type,cutoff_month_cat,orders_cnt_30d_cat,noise_00_30d,noise_01_30d,noise_02_30d,noise_03_30d,noise_04_30d,noise_05_30d,noise_06_30d,noise_07_30d,noise_08_30d,noise_09_30d,noise_10_30d,noise_11_30d,noise_12_30d,noise_13_30d,noise_14_30d,noise_15_30d,noise_16_30d,noise_17_30d,noise_18_30d,noise_19_30d,target_30d
i64,i64,f64,f64,f64,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,3,0.726454,1510.972845,1208.778276,"""c""","""android""","""01""","""many""",0.959765,0.582553,0.854714,-0.774324,2.239728,0.239302,1.304993,0.522514,-0.391284,0.443959,0.289423,1.621543,-0.198423,-0.44265,-0.601266,-0.213918,-0.623009,-0.613582,-0.328674,-0.660579,689.69474
2,0,-1.183848,0.0,0.0,"""c""","""android""","""01""","""few""",-0.444314,0.627592,0.864416,1.702637,0.908753,2.101582,-0.910223,0.944597,0.704447,1.986541,-0.252456,0.507674,0.408981,-2.842256,0.253346,0.614454,0.64733,-0.432868,-0.280047,-1.065422,0.0
3,1,0.929778,595.420544,476.336435,"""a""","""android""","""01""","""few""",0.571398,1.314346,-1.724585,-0.961408,-0.065572,-0.665514,0.898848,1.105293,1.007626,-0.010034,-0.673783,0.365692,0.243285,0.098092,1.358161,-0.395819,-0.494852,-0.030512,0.612907,0.475179,994.162003
4,2,0.394937,869.178985,695.343188,"""c""","""web""","""01""","""many""",-0.278728,0.415137,0.602878,-0.903025,0.108702,0.415556,-2.153185,1.212169,-0.830314,0.546205,-1.359738,-0.288275,-0.582481,0.575674,1.5507,1.468721,-0.912106,-1.680565,-1.629012,0.445815,443.764491
5,0,-1.024594,0.0,0.0,"""c""","""android""","""01""","""few""",-1.009818,-1.35749,0.39877,-0.215723,0.229193,1.495229,-0.387114,-0.377883,-0.519834,-0.366213,1.114486,-0.797148,-0.14032,0.906549,1.472401,-0.464066,-1.371233,1.546023,0.479901,1.285203,0.0


In [27]:
def demo_config(run_name, model, target, **extra):
    """Конфиг под 100 пользователей: правила боевые, объём счёта урезан."""
    return C.smoke_config(
        DEMO / "runs" / run_name, DEMO / "snapshots",
        users_limit=0, feature_limit=0, model_mode=model, target_mode=target,
        stage1=C.Stage1Config(actual_runs=2, null_runs=6, min_keep=5, max_keep=60,
                              null_iterations=60, actual_iterations=200,
                              actual_early_stopping=25),
        stage3=C.Stage3Config(speculation_width=6),
        parallel=C.ParallelConfig(n_workers=4, threads_per_worker=1,
                                  wide_threads_per_worker=1),
        shared_prepared_dir=str(DEMO / "prepared" / ("anchor" if "anchor_dir" in extra else "plain")),
        **extra)


import logging
import time

summary = []


def run_demo(run_name, model, target, **extra):
    started = time.perf_counter()
    report = run_selection(demo_config(run_name, model, target, **extra))
    logging.getLogger("nsel").handlers.clear()   # чтобы следующий прогон не дублировал вывод
    summary.append({"прогон": run_name, "модель": model, "таргет": target,
                    "признаков": len(report["selected_features"]),
                    "метрика": round(report["final_metric"], 5),
                    "секунд": round(time.perf_counter() - started, 1),
                    "набор": ", ".join(report["selected_features"])})
    return report


report = run_demo("catboost_log_level", "catboost", "log_level")

01:15:58 INFO    сочетание catboost_log_level, отпечаток 39abede2081a82f9


01:15:58 INFO    [подготовка] признаков 27: числовых 24, категориальных 3


01:15:58 INFO    [подготовка] раскладываю срез 2025-09-16


01:15:58 INFO    [подготовка] раскладываю срез 2025-10-16


01:15:58 INFO    [подготовка] раскладываю срез 2025-11-15


...

01:16:02 INFO    [stage6_backward] без gmv_30d                                    метрика +0.573352 изъятие ухудшает метрику -> False


01:16:02 INFO    [stage6_backward] удалено 0, осталось 2


01:16:02 INFO    стадия 6 заняла 0.0с


01:16:02 INFO    [stage7_optuna] выключено


01:16:02 INFO    итог: признаков 2, метрика 2.627446, файл demo_sintetika/runs/catboost_log_level/selected_features_catboost_log_level.json


Выше полный лог одного прогона: подготовка, все стадии и итог. Строки
`заданий N, из кэша M, к счёту K` показывают батчи и кэш, строки стадии 3 — решения по
каждому кандидату. Остальные сочетания прогоняем без лишнего вывода и сводим в таблицу.

In [28]:
import warnings
warnings.filterwarnings("ignore")   # tqdm и experimental-предупреждения Optuna

logging.disable(logging.INFO)
run_demo("lightgbm_log_diff", "lightgbm", "log_diff")
run_demo("xgboost_log_level", "xgboost", "log_level")
run_demo("catboost_segment_adaptive", "catboost", "segment_adaptive",
         anchor_dir=str(DEMO / "anchors"))
run_demo("xgboost_segment_logspace", "xgboost", "segment_logspace",
         anchor_dir=str(DEMO / "anchors"))
run_demo("lightgbm_log_ratio_ewma", "lightgbm", "log_ratio_ewma")
run_demo("catboost_optuna", "catboost", "log_level",
         optuna=C.OptunaConfig(enabled=True, exploration_trials=6, focused_trials=4,
                               tpe_startup_trials_explore=3, tpe_startup_trials_focus=2))
# Повтор первого прогона: всё должно взяться из кэша
run_demo("catboost_log_level", "catboost", "log_level")
logging.disable(logging.NOTSET)
pd.DataFrame(summary)

,прогон,модель,таргет,признаков,метрика,секунд,набор
0,catboost_log_level,catboost,log_level,2,2.62745,4.1,"gmv_30d, orders_cnt_30d"
1,lightgbm_log_diff,lightgbm,log_diff,2,2.70783,5.1,"gmv_30d, orders_cnt_30d"
2,xgboost_log_level,xgboost,log_level,2,2.43582,5.0,"gmv_30d, habit_score_90d"
3,catboost_segment_adaptive,catboost,segment_adaptive,2,2.26877,4.0,"gmv_30d, habit_score_90d"
4,xgboost_segment_logspace,xgboost,segment_logspace,2,2.36575,4.9,"habit_score_90d, orders_cnt_30d"
5,lightgbm_log_ratio_ewma,lightgbm,log_ratio_ewma,2,2.70482,5.0,"gmv_30d, orders_cnt_30d"
6,catboost_optuna,catboost,log_level,2,2.62745,7.4,"gmv_30d, orders_cnt_30d"
7,catboost_log_level,catboost,log_level,2,2.62745,0.0,"gmv_30d, orders_cnt_30d"


In [29]:
# Проверки: шумовые признаки не прошли, биннинговая категория отброшена, повтор взят из кэша
manifest = json.loads((DEMO / "prepared" / "plain" / "manifest.json").read_text())
assert "orders_cnt_30d_cat" not in manifest["features"], "биннинговая категория не отброшена"
assert "cutoff_month_cat" in manifest["features"], "календарная категория потерялась"
for row in summary:
    assert row["признаков"] > 0
    assert "noise_" not in row["набор"], f"шум в наборе {row['прогон']}"
assert summary[-1]["набор"] == summary[0]["набор"] and summary[-1]["метрика"] == summary[0]["метрика"]
assert summary[-1]["секунд"] < 1.0, "повтор не взялся из кэша"
print("признаков в пуле:", len(manifest["features"]))
print("все проверки пройдены")

признаков в пуле: 27
все проверки пройдены


### Все ветки кода и обрыв с продолжением

С боевыми порогами на 100 пользователях стадия 3 заканчивается за пару решений, и
парное спасение, разбор сомнительных и обратная чистка ничего не делают. Поэтому здесь
пороги нарочно ослаблены: кандидат принимается при любом положительном выигрыше,
пары — при любом, все принятые считаются сомнительными, а чистка снимает признак, если
без него метрика хуже не больше чем на 0.02. Шум в таком наборе ожидаем, это проверка
кода, а не качества.

Сначала прогон без обрыва. Потом такой же прогон, который обрывается файлом `STOP`
посреди стадии 3 и запускается снова. Он должен продолжить с того же места и прийти к
тому же итогу.

In [30]:
import threading
from nsel import cache, orchestrator
from nsel.pool import StopRequested


def permissive_config(run_name):
    """Пороги ослаблены так, чтобы каждая стадия что-то делала."""
    return C.smoke_config(
        DEMO / "runs" / run_name, DEMO / "snapshots", users_limit=0, feature_limit=0,
        model_mode="catboost", target_mode="log_level",
        stage1=C.Stage1Config(actual_runs=2, null_runs=6, min_keep=27, max_keep=60,
                              null_iterations=60, actual_iterations=200,
                              actual_early_stopping=25),
        stage3=C.Stage3Config(min_improvement=1e-9, min_improved_folds=1,
                              max_allowed_fold_degradation=1.0,
                              max_zero_segment_degradation=1.0,
                              max_positive_segment_degradation=1.0,
                              rerank_after_n_added=3, speculation_width=6),
        stage4=C.Stage4Config(confirm_doubtful=True, doubtful_improvement_factor=1e9),
        stage5=C.Stage5Config(near_miss_fraction=-1e9, min_improvement=-1.0,
                              min_improved_folds=0, max_allowed_fold_degradation=1.0),
        backward=C.BackwardConfig(neutral_tolerance=0.02),
        parallel=C.ParallelConfig(n_workers=4, threads_per_worker=1,
                                  wide_threads_per_worker=1),
        shared_prepared_dir=str(DEMO / "prepared" / "plain"))


logging.disable(logging.INFO)
reference = run_selection(permissive_config("bez_obryva"))
logging.getLogger("nsel").handlers.clear()
print("стадия 3 приняла:      ", reference["forward_selected"])
print("парное спасение:       ", reference["pair_rescue"]["added"])
print("сомнительных проверено:", len(reference["multiseed"]["confirmations"]),
      "снято:", reference["multiseed"]["dropped"])
print("обратная чистка сняла: ", reference["backward_removed"])
print("итог:                  ", reference["selected_features"])
assert reference["pair_rescue"]["added"], "парное спасение ничего не добавило"
assert reference["multiseed"]["confirmations"], "сомнительные не проверялись"
assert reference["backward_removed"], "обратная чистка ничего не сняла"

стадия 3 приняла:       ['gmv_30d', 'orders_cnt_30d', 'habit_score_90d', 'noise_15_30d', 'segment_kind', 'device_type']
парное спасение:        [['noise_04_30d', 'noise_11_30d'], ['noise_10_30d', 'noise_03_30d']]
сомнительных проверено: 5 снято: []
обратная чистка сняла:  ['noise_03_30d', 'noise_10_30d', 'noise_15_30d', 'orders_cnt_30d']
итог:                   ['gmv_30d', 'habit_score_90d', 'segment_kind', 'device_type', 'noise_04_30d', 'noise_11_30d']


In [31]:
config = permissive_config("s_obryvom")
stop_file = Path(config.run_dir) / "STOP"
STAGE3 = "stage3_ranked_forward"


def stop_inside_stage3():
    """Кладёт STOP, когда стадия 3 посчитала четыре кандидата (по три фолда)."""
    while True:
        time.sleep(0.01)
        try:
            if cache.count_iterations(config.cache_db, STAGE3) >= 12:
                stop_file.touch()
                return
        except Exception:
            pass   # база ещё не создана


threading.Thread(target=stop_inside_stage3, daemon=True).start()
try:
    run_selection(config)
    stopped = False
except StopRequested as error:
    stopped = True
    print("оборван на стадии", error)
logging.getLogger("nsel").handlers.clear()
assert stopped, "обрыва не случилось"
before = cache.count_iterations(config.cache_db, STAGE3)
state = cache.get_stage_state(config.cache_db, STAGE3, "progress")
print(f"в кэше обучений стадии 3: {before}; сохранено состояние: "
      f"отобрано {len(state['selected'])}, осталось кандидатов {len(state['candidates'])}")

stop_file.unlink(missing_ok=True)
orchestrator._STOP["requested"] = False
resumed = run_selection(config)
logging.getLogger("nsel").handlers.clear()
logging.disable(logging.NOTSET)
after = cache.count_iterations(config.cache_db, STAGE3)
print(f"после продолжения обучений стадии 3 в кэше: {after} (из них {before} посчитаны до обрыва)")
assert resumed["selected_features"] == reference["selected_features"]
assert resumed["final_metric"] == reference["final_metric"]
print("продолжение пришло к тому же итогу, что и прогон без обрыва:",
      resumed["selected_features"])

оборван на стадии stage3_ranked_forward
в кэше обучений стадии 3: 13; сохранено состояние: отобрано 3, осталось кандидатов 23


после продолжения обучений стадии 3 в кэше: 84 (из них 13 посчитаны до обрыва)
продолжение пришло к тому же итогу, что и прогон без обрыва: ['gmv_30d', 'habit_score_90d', 'segment_kind', 'device_type', 'noise_04_30d', 'noise_11_30d']


## 5. Запуск на настоящих данных

Нужен каталог со срезами `snapshot_YYYYMMDD.parquet` на даты 2025-09-16, 2025-10-16,
2025-11-15, 2025-12-15 и 2026-01-14. В срезах обязательны столбцы
`__snapshot_index_user_id__`, `target_30d` и `gmv_30d`, всё остальное считается признаками.
Строковые столбцы считаются категориальными.

Для адаптивных таргетов нужен ещё `anchor_dir` с файлами `anchor_segment_adaptive_YYYYMMDD.parquet`
(их строит `tools/build_adaptive_anchor.py`).

Из ноутбука:
```python
config = C.RunConfig(run_dir=Path("runs/catboost_log_level"),
                     snapshot_dir=Path("/путь/к/срезам"),
                     model_mode="catboost", target_mode="log_level",
                     parallel=C.ParallelConfig(n_workers=15, threads_per_worker=2))
report = run_selection(config)
```

Из командной строки (в папке `kod/`):
```bash
# быстрая проверка на подвыборке, около минуты
python run_nsel.py --run-dir runs/smoke --snapshot-dir /путь/к/срезам --smoke --users 3000 --features 30
# боевой прогон одного сочетания
python run_nsel.py --run-dir runs/catboost_log_level --snapshot-dir /путь/к/срезам \
    --model catboost --target log_level --workers 30 --threads 2
```

На полном объёме (250 тыс. пользователей, 909 признаков) одно сочетание считалось на сервере
от 1 до 3 часов. Остановить можно файлом `STOP` в `run-dir`. Повторный запуск с тем же
`run-dir` продолжит с места остановки.

## 6. Сверка с `kod/`

In [32]:
# Каждый файл, записанный ноутбуком, сравнивается с оригиналом из kod/ (если папка рядом)
import hashlib

original = Path("kod")
if original.exists():
    mismatched = []
    written = sorted(path for path in Path("nsel_iz_noutbuka").rglob("*.py") if "__pycache__" not in path.parts)
    for path in written:
        twin = original / path.relative_to("nsel_iz_noutbuka")
        if hashlib.md5(path.read_bytes()).hexdigest() != hashlib.md5(twin.read_bytes()).hexdigest():
            mismatched.append(str(twin))
    print(f"файлов {len(written)}, расхождений {len(mismatched)}", mismatched or "")
else:
    print("папки kod/ рядом нет")

файлов 21, расхождений 0 
